# 4.7.1 Package Forecasting System for Counterfactual and Showcase Use

Chapter 4 has completed the forecasting model-development and evaluation work. The remaining task is to package that work so two very different downstream consumers can use it safely.

**Chapter 5** needs the frozen model-family definitions and the already-established Pre-CP forecasting contracts so it can run a deliberately small, pre-policy-only family comparison before constructing the no-congestion-pricing reference path.

The **Showcase** needs the frozen final-holdout evaluation surface, a longer champion-forecast history for interactive exploration, and the reliability context needed to explain where the system performed well or struggled.

This notebook therefore performs **packaging, validation, and handoff only**. It does not reopen model development, select the Chapter 5 Pre-CP Reference winner, fit a counterfactual model, or generate the no-congestion-pricing forecast path.

### Notebook contract

The handoff is intentionally split into two products.

**Counterfactual handoff**

* Preserve one finalized candidate configuration from each of the four Chapter 4 forecasting families for every Target × Horizon job.
* Preserve Chapter 4 champion and runner-up identities as provenance, not as the Chapter 5 selection rule.
* Package the existing Pre-CP validation, support, benchmark, feature, refit, recursion, and application contracts.
* Leave the Chapter 5 Pre-CP Reference roster unselected.

**Showcase handoff**

* Preserve the benchmark-comparable final-holdout surface used for Chapter 4 evaluation.
* Add a longer champion-forecast history from April 5, 2025 through March 31, 2026 for interactive exploration, with ordinary validation and final holdout labeled separately.
* Attach the completed 4.6.2 reliability and failure context to the final-holdout surface without changing the frozen forecast of record.
* Precompute compact final-holdout summaries for common forecasting views.
* Support scale-aware Forecast Wins & Misses while preserving native-unit observations and predictions for interpretation.

Large upstream matrices and recursive templates remain at their authoritative locations. 4.7.1 references them rather than creating unnecessary duplicate copies.

In [1]:
# ---------------------------------------------------------------------
# 4.7.1.0A — Imports
# ---------------------------------------------------------------------

import os
import uuid
from pathlib import Path
import shutil

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from IPython.display import display

In [2]:
# ---------------------------------------------------------------------
# 4.7.1.0B — Frozen project constants and analytical contracts
# ---------------------------------------------------------------------

PROJECT_ROOT = Path.cwd()
PIPELINE_DIR = PROJECT_ROOT / "pipeline_data"

FORECAST_TARGETS = [
    "taxi_trip_count",
    "taxi_avg_trip_speed",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_speed",
    "subway_ridership",
]

FORECAST_HORIZONS = [1, 2, 5]

TARGET_LABELS = {
    "taxi_trip_count": "Taxi trips",
    "taxi_avg_trip_speed": "Taxi speed",
    "fhvhv_trip_count": "FHVHV trips",
    "fhvhv_avg_trip_speed": "FHVHV speed",
    "subway_ridership": "Subway ridership",
}

FAMILY_ORDER = [
    "classical",
    "tree",
    "neural",
    "transformer",
]

FAMILY_LABELS = {
    "classical": "Classical",
    "tree": "Tree",
    "neural": "Neural",
    "transformer": "Transformer",
}

FAMILY_REPRESENTATIVES = {
    "classical": "SARIMAX",
    "tree": "LightGBM",
    "neural": "Lag-window MLP",
    "transformer": "PatchTST-inspired Transformer",
}

EXPECTED_JOB_COUNT = 15
EXPECTED_FAMILY_CANDIDATE_ROWS = 60

EXPECTED_PRE_CP_NATIVE_SEQUENCES = 1_039
EXPECTED_PRIMARY_CROSS_REGIME_SEQUENCES = 1_029
EXPECTED_SHOWCASE_ROWS = 1_421_436

PRE_CP_SELECTION_END_DATE = pd.Timestamp("2024-12-31")
PRE_CP_REFIT_ONLY_START_DATE = pd.Timestamp("2025-01-01")
PRE_CP_FINAL_REFIT_END_DATE = pd.Timestamp("2025-01-04")
COUNTERFACTUAL_START_DATE = pd.Timestamp("2025-01-05")
COUNTERFACTUAL_END_DATE = pd.Timestamp("2026-03-31")
SHOWCASE_HISTORY_VALIDATION_START_DATE = pd.Timestamp("2025-04-05")
SHOWCASE_HISTORY_VALIDATION_END_DATE = pd.Timestamp("2026-01-04")
SHOWCASE_HISTORY_HOLDOUT_START_DATE = pd.Timestamp("2026-01-05")
SHOWCASE_HISTORY_HOLDOUT_END_DATE = pd.Timestamp("2026-03-31")

PRE_CP_FINAL_REFIT_SEQUENCE_ID = 3_675
COUNTERFACTUAL_START_SEQUENCE_ID = 3_676

COMMON_FORECAST_KEY = [
    "metric",
    "horizon",
    "forecast_sequence_id",
    "taxi_zone_id",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
]

PHYSICAL_FORECAST_KEY = [
    "modeling_track",
    *COMMON_FORECAST_KEY,
]

CROSS_ARTIFACT_ATOL = 1e-9

SHOWCASE_SURFACE_CONTRACT_ID = (
    "showcase_forecast_record_surface_v1"
)

SHOWCASE_HISTORY_CONTRACT_ID = (
    "showcase_forecast_history_v1"
)

In [3]:
# ---------------------------------------------------------------------
# 4.7.1.0C — Authoritative upstream directories and paths
# ---------------------------------------------------------------------

INPUT_411_FINAL_DIR = PIPELINE_DIR / "4.1.1.final_tables"
INPUT_412_FINAL_DIR = PIPELINE_DIR / "4.1.2.final_tables"
INPUT_421_FINAL_DIR = PIPELINE_DIR / "4.2.1.final_tables"
WEEKLY_STL_SELECTION_REGISTRY_PATH = (
    INPUT_421_FINAL_DIR / "weekly_stl_selection_registry.parquet"
)
WEEKLY_STL_TEMPLATE_INTERVAL_REGISTRY_PATH = (
    INPUT_421_FINAL_DIR / "weekly_stl_template_interval_registry.parquet"
)
INPUT_422_FINAL_DIR = PIPELINE_DIR / "4.2.2.final_tables"
INPUT_423_FINAL_DIR = PIPELINE_DIR / "4.2.3.final_tables"
INPUT_424_FINAL_DIR = PIPELINE_DIR / "4.2.4.final_tables"

INPUT_431_FINAL_DIR = PIPELINE_DIR / "4.3.1.final_tables"
INPUT_441_FINAL_DIR = PIPELINE_DIR / "4.4.1.final_tables"
INPUT_451_FINAL_DIR = PIPELINE_DIR / "4.5.1.final_tables"
INPUT_452_FINAL_DIR = PIPELINE_DIR / "4.5.2.final_tables"
INPUT_461_FINAL_DIR = PIPELINE_DIR / "4.6.1.final_tables"
INPUT_462_FINAL_DIR = PIPELINE_DIR / "4.6.2.final_tables"


# ---------------------------------------------------------------------
# 4.1.1 — Stable Forecast Sequence geography
# ---------------------------------------------------------------------
# This compact registry carries the canonical Taxi Zone labels attached to
# each Forecast Sequence. The longer Showcase history uses it so every model
# family receives the same zone and borough identity.

FORECAST_SEQUENCE_SUPPORT_REGISTRY_PATH = (
    INPUT_411_FINAL_DIR
    / "forecasting_sequence_support_registry.parquet"
)

# ---------------------------------------------------------------------
# 4.1.2 — Time, population, and modeling-track contracts
# ---------------------------------------------------------------------

ROLLING_ORIGIN_PATH = (
    INPUT_412_FINAL_DIR
    / "rolling_origin_registry.parquet"
)

TRACK_CONTRACT_PATH = (
    INPUT_412_FINAL_DIR
    / "forecasting_track_contract_registry.parquet"
)

PRE_CP_REFIT_APPLICATION_CONTRACT_PATH = (
    INPUT_412_FINAL_DIR
    / "pre_cp_refit_application_contract.parquet"
)

PRE_CP_SEQUENCE_REGISTRY_PATH = (
    INPUT_412_FINAL_DIR
    / "pre_cp_reference_sequence_registry.parquet"
)

CROSS_REGIME_SEQUENCE_REGISTRY_PATH = (
    INPUT_412_FINAL_DIR
    / "cross_regime_comparison_sequence_registry.parquet"
)


# ---------------------------------------------------------------------
# 4.2.1 — Temporal features and recursive no-CP state contracts
# ---------------------------------------------------------------------

FORECAST_TARGET_CONTEXT_PATH = (
    INPUT_421_FINAL_DIR
    / "forecast_target_calendar_context.parquet"
)

PRE_CP_COUNTERFACTUAL_CALENDAR_PATH = (
    INPUT_421_FINAL_DIR
    / "pre_cp_counterfactual_calendar.parquet"
)

PRE_CP_COUNTERFACTUAL_WEEKLY_STATE_PATH = (
    INPUT_421_FINAL_DIR
    / "pre_cp_counterfactual_weekly_state.parquet"
)

PRE_CP_RECURSIVE_FEATURE_CONTRACT_PATH = (
    INPUT_421_FINAL_DIR
    / "pre_cp_recursive_feature_contract.parquet"
)


# ---------------------------------------------------------------------
# 4.2.2 — Multimodal/spatial post-freeze context behavior
# ---------------------------------------------------------------------

PRE_CP_CONTEXT_RUNTIME_CONTRACT_PATH = (
    INPUT_422_FINAL_DIR
    / "pre_cp_context_runtime_contract.parquet"
)


# ---------------------------------------------------------------------
# 4.2.3 — Complete forecasting matrices and runtime scaffold
# ---------------------------------------------------------------------

COMPLETE_FEATURE_CATALOG_PATH = (
    INPUT_423_FINAL_DIR
    / "forecast_feature_catalog_complete.parquet"
)

PRE_CP_FORECASTING_MATRIX_PATH = (
    INPUT_423_FINAL_DIR
    / "forecasting_matrix__pre_cp_reference.parquet"
)

PRE_CP_FINAL_REFIT_SUPPLEMENT_PATH = (
    INPUT_423_FINAL_DIR
    / "pre_cp_final_refit_training_supplement.parquet"
)

PRE_CP_FINAL_RUNTIME_CONTRACT_PATH = (
    INPUT_423_FINAL_DIR
    / "pre_cp_final_runtime_contract.parquet"
)

PRE_CP_COUNTERFACTUAL_TEMPLATE_PATH = (
    INPUT_423_FINAL_DIR
    / "pre_cp_counterfactual_application_template.parquet"
)

FINAL_FORECASTING_MATRIX_QA_PATH = (
    INPUT_423_FINAL_DIR
    / "final_forecasting_matrix_qa.parquet"
)

FORECASTING_MATRIX_MANIFEST_PATH = (
    INPUT_423_FINAL_DIR
    / "forecasting_matrix_manifest.parquet"
)


# ---------------------------------------------------------------------
# 4.2.4 — Frozen simple benchmark and Pre-CP row-level handoff
# ---------------------------------------------------------------------

ADVANCED_BENCHMARK_REGISTRY_PATH = (
    INPUT_424_FINAL_DIR
    / "advanced_model_benchmark_registry.parquet"
)

SELECTED_PRE_CP_BENCHMARK_PREDICTIONS_PATH = (
    INPUT_424_FINAL_DIR
    / "selected_benchmark_predictions__pre_cp_reference.parquet"
)

BASELINE_FORECASTABILITY_QA_PATH = (
    INPUT_424_FINAL_DIR
    / "final_baseline_forecastability_qa.parquet"
)

BASELINE_FORECASTABILITY_MANIFEST_PATH = (
    INPUT_424_FINAL_DIR
    / "baseline_forecastability_manifest.parquet"
)


# ---------------------------------------------------------------------
# 4.6.1 / 4.6.2 — Evaluation handoffs
# ---------------------------------------------------------------------

FORECAST_EVALUATION_HANDOFF_461_PATH = (
    INPUT_461_FINAL_DIR
    / "forecast_evaluation_handoff_manifest.parquet"
)

FORECAST_EVALUATION_HANDOFF_462_PATH = (
    INPUT_462_FINAL_DIR
    / "forecast_evaluation_handoff_manifest.parquet"
)

# ---------------------------------------------------------------------
# Ordinary-validation forecasts used by the longer Showcase history
# ---------------------------------------------------------------------
# These are the final family-owned row-level handoffs already used in the
# Chapter 4 cross-family comparison. No model is rerun here.

TREE_VALIDATION_PREDICTIONS_PATH = (
    INPUT_441_FINAL_DIR
    / "lgbm_selected_row_level_validation_predictions.parquet"
)

NEURAL_VALIDATION_PREDICTIONS_PATH = (
    INPUT_451_FINAL_DIR
    / "neural_selected_validation_predictions.parquet"
)

TRANSFORMER_VALIDATION_PREDICTIONS_PATH = (
    INPUT_452_FINAL_DIR
    / "transformer_selected_validation_predictions.parquet"
)

In [4]:
# ---------------------------------------------------------------------
# 4.7.1.0D — Output paths
# ---------------------------------------------------------------------

INTERMEDIATE_DIR = PIPELINE_DIR / "4.7.1.intermediate_tables"
FINAL_DIR = PIPELINE_DIR / "4.7.1.final_tables"

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Chapter 5-facing package
# ---------------------------------------------------------------------

COUNTERFACTUAL_CANDIDATE_REGISTRY_PATH = (
    FINAL_DIR
    / "counterfactual_candidate_registry.parquet"
)

MODEL_CONTRACT_MANIFEST_PATH = (
    FINAL_DIR
    / "counterfactual_model_contract_manifest.parquet"
)

CHAPTER4_SELECTION_PROVENANCE_PATH = (
    FINAL_DIR
    / "chapter4_model_selection_provenance.parquet"
)

PRE_CP_SEQUENCE_SUPPORT_REGISTRY_PATH = (
    FINAL_DIR
    / "pre_cp_reference_sequence_support_registry.parquet"
)

PRE_CP_BENCHMARK_REGISTRY_PATH = (
    FINAL_DIR
    / "pre_cp_reference_benchmark_registry.parquet"
)

PRE_CP_HANDOFF_MANIFEST_PATH = (
    FINAL_DIR
    / "pre_cp_reference_handoff_manifest.parquet"
)


# ---------------------------------------------------------------------
# Showcase-facing package
# ---------------------------------------------------------------------

SHOWCASE_FORECAST_RECORD_SURFACE_PATH = (
    FINAL_DIR
    / "showcase_forecast_record_surface.parquet"
)

SHOWCASE_FORECAST_RECORD_AUDIT_PATH = (
    INTERMEDIATE_DIR
    / "showcase_forecast_record_surface_audit.parquet"
)

SHOWCASE_FORECAST_HISTORY_SURFACE_PATH = (
    FINAL_DIR
    / "showcase_forecast_history_surface.parquet"
)

SHOWCASE_FORECAST_HISTORY_AUDIT_PATH = (
    INTERMEDIATE_DIR
    / "showcase_forecast_history_surface_audit.parquet"
)

SHOWCASE_JOB_SUMMARY_PATH = (
    FINAL_DIR
    / "showcase_forecast_job_summary.parquet"
)

SHOWCASE_ZONE_SUMMARY_PATH = (
    FINAL_DIR
    / "showcase_forecast_zone_summary.parquet"
)

SHOWCASE_TEMPORAL_BUCKET_SUMMARY_PATH = (
    FINAL_DIR
    / "showcase_forecast_temporal_bucket_summary.parquet"
)

SHOWCASE_DATE_SUMMARY_PATH = (
    FINAL_DIR
    / "showcase_forecast_date_summary.parquet"
)

SHOWCASE_HANDOFF_MANIFEST_PATH = (
    FINAL_DIR
    / "showcase_forecasting_handoff_manifest.parquet"
)
SHOWCASE_PORTABLE_MANIFEST_PATH = (
    FINAL_DIR
    / "showcase_forecasting_portable_manifest.parquet"
)

# ---------------------------------------------------------------------
# Complete Chapter 4 forecasting handoff
# ---------------------------------------------------------------------

FINAL_HANDOFF_QA_PATH = (
    FINAL_DIR
    / "forecasting_system_handoff_qa.parquet"
)

FINAL_HANDOFF_MANIFEST_PATH = (
    FINAL_DIR
    / "forecasting_system_handoff_manifest.parquet"
)

In [5]:
# ---------------------------------------------------------------------
# 4.7.1.0E — Frozen family finalist handoffs
# ---------------------------------------------------------------------

FAMILY_HANDOFFS = {
    "classical": {
        "registry":
            INPUT_431_FINAL_DIR
            / "classical_final_winner_registry.parquet",
        "qa":
            INPUT_431_FINAL_DIR
            / "final_classical_model_selection_qa.parquet",
        "manifest":
            INPUT_431_FINAL_DIR
            / "classical_model_selection_manifest.parquet",
        "registry_track_column": "modeling_track",
        "configuration_columns": [
            "model_family",
            "candidate_label",
            "representation",
            "refit_cadence",
        ],
    },

    "tree": {
        "registry":
            INPUT_441_FINAL_DIR
            / "lgbm_selected_model_registry.parquet",
        "qa":
            INPUT_441_FINAL_DIR
            / "lgbm_selected_qa.parquet",
        "manifest":
            INPUT_441_FINAL_DIR
            / "lgbm_selected_manifest.parquet",
        "registry_track_column": "modeling_track",
        "configuration_columns": [
            "model_family",
            "production_config_id",
            "capacity_profile",
            "target_representation",
            "refit_cadence",
        ],
    },

    "neural": {
        "registry":
            INPUT_451_FINAL_DIR
            / "neural_final_model_registry.parquet",
        "qa":
            INPUT_451_FINAL_DIR
            / "neural_closeout_qa.parquet",
        "manifest":
            INPUT_451_FINAL_DIR
            / "neural_handoff_manifest.parquet",
        "registry_track_column": "modeling_track",
        "configuration_columns": [
            "model_family",
            "model_definition_id",
            "profile_id",
            "refit_cadence",
        ],
    },

    "transformer": {
        "registry":
            INPUT_452_FINAL_DIR
            / "transformer_final_model_registry.parquet",
        "qa":
            INPUT_452_FINAL_DIR
            / "transformer_closeout_qa.parquet",
        "manifest":
            INPUT_452_FINAL_DIR
            / "transformer_handoff_manifest.parquet",
        "registry_track_column": "track",
        "configuration_columns": [
            "model_family",
            "model_definition_id",
            "representation_id",
            "profile_id",
            "refit_cadence",
        ],
    },
}

# ---------------------------------------------------------------------
# Ordinary-validation inputs for the longitudinal Showcase surface
# ---------------------------------------------------------------------
# The three families below own all 15 ordinary Chapter 4 champions.
# Their row-level contracts are already frozen; only the Neural handoff uses
# a different name for the observed target.

SHOWCASE_HISTORY_SOURCES = {
    "tree": {
        "path": TREE_VALIDATION_PREDICTIONS_PATH,
        "actual_column": "actual",
    },
    "neural": {
        "path": NEURAL_VALIDATION_PREDICTIONS_PATH,
        "actual_column": "target_observed_value",
    },
    "transformer": {
        "path": TRANSFORMER_VALIDATION_PREDICTIONS_PATH,
        "actual_column": "actual",
    },
}


In [6]:
# ---------------------------------------------------------------------
# 4.7.1.0F — Family-specific executable contract references
# ---------------------------------------------------------------------

MODEL_CONTRACT_SPECS = [
    (
        "classical",
        "Final SARIMAX model contract",
        INPUT_431_FINAL_DIR / "sarimax_final_model_contract.parquet",
        "parquet",
        "reuse",
    ),
    (
        "classical",
        "SARIMAX exogenous-feature contract",
        INPUT_431_FINAL_DIR / "sarimax_exog_feature_contract.parquet",
        "parquet",
        "reuse",
    ),
    (
        "classical",
        "Classical final QA",
        INPUT_431_FINAL_DIR / "final_classical_model_selection_qa.parquet",
        "parquet",
        "audit",
    ),

    (
        "tree",
        "Selected LightGBM model registry",
        INPUT_441_FINAL_DIR / "lgbm_selected_model_registry.parquet",
        "parquet",
        "reuse",
    ),
    (
        "tree",
        "Ordered LightGBM predictor registry",
        INPUT_441_FINAL_DIR / "lgbm_predictor_registry.parquet",
        "parquet",
        "reuse",
    ),
    (
        "tree",
        "LightGBM forecast-output contract",
        INPUT_441_FINAL_DIR / "lgbm_forecast_output_contract.parquet",
        "parquet",
        "reuse",
    ),
    (
        "tree",
        "Selected LightGBM QA",
        INPUT_441_FINAL_DIR / "lgbm_selected_qa.parquet",
        "parquet",
        "audit",
    ),

    (
        "neural",
        "Neural model-definition contract",
        INPUT_451_FINAL_DIR / "neural_model_definition_contract.parquet",
        "parquet",
        "reuse",
    ),
    (
        "neural",
        "Neural input registry",
        INPUT_451_FINAL_DIR / "neural_input_registry.parquet",
        "parquet",
        "reuse",
    ),
    (
        "neural",
        "Neural sequence/preprocessing contract",
        INPUT_451_FINAL_DIR / "neural_sequence_preprocessing_contract.parquet",
        "parquet",
        "reuse",
    ),
    (
        "neural",
        "Shared Neural/Transformer mobility-history state",
        INPUT_451_FINAL_DIR / "neural_mobility_history_state.npz",
        "npz",
        "reuse",
    ),
    (
        "neural",
        "Neural forecast-output contract",
        INPUT_451_FINAL_DIR / "neural_forecast_output_contract.parquet",
        "parquet",
        "reuse",
    ),
    (
        "neural",
        "Neural training-policy evidence",
        INPUT_451_FINAL_DIR / "neural_training_policy_evidence.parquet",
        "parquet",
        "reuse",
    ),
    (
        "neural",
        "Neural closeout QA",
        INPUT_451_FINAL_DIR / "neural_closeout_qa.parquet",
        "parquet",
        "audit",
    ),

    (
        "transformer",
        "Transformer model-definition contract",
        INPUT_452_FINAL_DIR / "transformer_model_definition_contract.parquet",
        "parquet",
        "reuse",
    ),
    (
        "transformer",
        "Transformer context contract",
        INPUT_452_FINAL_DIR / "transformer_context_contract.parquet",
        "parquet",
        "reuse",
    ),
    (
        "transformer",
        "Transformer input registry",
        INPUT_452_FINAL_DIR / "transformer_input_registry.parquet",
        "parquet",
        "reuse",
    ),
    (
        "transformer",
        "Transformer forecast-output contract",
        INPUT_452_FINAL_DIR / "transformer_forecast_output_contract.parquet",
        "parquet",
        "reuse",
    ),
    (
        "transformer",
        "Transformer training-policy evidence",
        INPUT_452_FINAL_DIR / "transformer_training_policy_evidence.parquet",
        "parquet",
        "reuse",
    ),
    (
        "transformer",
        "Transformer closeout QA",
        INPUT_452_FINAL_DIR / "transformer_closeout_qa.parquet",
        "parquet",
        "audit",
    ),
]

In [7]:
# ---------------------------------------------------------------------
# 4.7.1.0G — Execution controls
# ---------------------------------------------------------------------

WRITE_FINAL_OUTPUTS = True

# The existing final-holdout surface is already frozen and normally recovered
# from its audit marker. The new longitudinal history has its own independent
# restart switch so one artifact never forces a rebuild of the other.
RUN_SHOWCASE_SURFACE_BUILD = True
REBUILD_SHOWCASE_SURFACE = False

RUN_SHOWCASE_HISTORY_BUILD = True
REBUILD_SHOWCASE_HISTORY = False

REBUILD_SHOWCASE_SUMMARIES = False

In [8]:
# ---------------------------------------------------------------------
# 4.7.1.0H — Validation, recovery, and persistence helpers
# ---------------------------------------------------------------------

def require_file(path, label):
    """Fail clearly when an authoritative upstream artifact is missing."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"{label} is missing: {path}")

    return path


def require_columns(frame, required_columns, label):
    """Fail when a loaded table no longer matches its expected contract."""
    missing = sorted(set(required_columns) - set(frame.columns))

    if missing:
        raise KeyError(f"{label} is missing required columns: {missing}")


def require_parquet_columns(path, required_columns, label):
    """Validate a large Parquet schema without loading its rows."""
    actual = set(pq.ParquetFile(path).schema_arrow.names)
    missing = sorted(set(required_columns) - actual)

    if missing:
        raise KeyError(f"{label} is missing required columns: {missing}")


def parquet_metadata(path):
    """Return compact physical metadata without materializing table rows."""
    path = Path(path)
    parquet_file = pq.ParquetFile(path)

    return {
        "rows": int(parquet_file.metadata.num_rows),
        "columns": int(len(parquet_file.schema_arrow.names)),
        "size_bytes": int(path.stat().st_size),
        "size_mb": path.stat().st_size / (1024 ** 2),
        "modified_ns": int(path.stat().st_mtime_ns),
        "schema_signature": "||".join(parquet_file.schema_arrow.names),
    }


def atomic_write_parquet(frame, path):
    """Write a completed Parquet atomically without exposing a partial final file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_name(
        f".{path.stem}.{uuid.uuid4().hex[:8]}.tmp.parquet"
    )

    frame.to_parquet(
        temporary_path,
        index=False,
        compression="zstd",
    )
    os.replace(temporary_path, path)

def qa_file_passes(path):
    """
    Interpret the final QA formats used by the four forecasting families.

    Classical stores PASS/FAIL text in `status`; the other finalized handoffs
    use a Boolean `passed` field.
    """
    qa_df = pd.read_parquet(path)

    if "passed" in qa_df.columns:
        return bool(qa_df["passed"].fillna(False).astype(bool).all()), qa_df

    if "status" in qa_df.columns:
        return bool(qa_df["status"].eq("PASS").all()), qa_df

    raise KeyError(
        f"{Path(path).name}: expected either a `passed` or `status` field."
    )


def expected_target_horizon_pairs():
    """Return the frozen 15 Target × Horizon jobs."""
    return {
        (metric, int(horizon))
        for metric in FORECAST_TARGETS
        for horizon in FORECAST_HORIZONS
    }


def normalize_finalist_registry(family, frame):
    """
    Normalize only the fields needed for the common Chapter 5 candidate index.

    Family-specific model definitions remain authoritative in their own
    contracts; this function deliberately does not flatten hyperparameters.
    """
    spec = FAMILY_HANDOFFS[family]
    track_column = spec["registry_track_column"]

    required_columns = [
        "metric",
        "horizon",
        track_column,
        *spec["configuration_columns"],
    ]
    require_columns(
        frame,
        required_columns,
        f"{FAMILY_LABELS[family]} finalist registry",
    )

    normalized = frame.copy().rename(
        columns={track_column: "source_modeling_track"}
    )
    normalized["family"] = family
    normalized["Family"] = FAMILY_LABELS[family]
    normalized["Representative"] = FAMILY_REPRESENTATIVES[family]
    normalized["horizon"] = normalized["horizon"].astype(int)

    normalized["source_configuration_id"] = (
        normalized[spec["configuration_columns"]]
        .astype(str)
        .agg(" · ".join, axis=1)
    )

    return normalized


def read_handoff_manifest(path, required_ids, label):
    """Load a producer-owned manifest and require its advertised artifact IDs."""
    frame = pd.read_parquet(path)
    require_columns(frame, ["artifact_id", "path"], label)

    if frame["artifact_id"].duplicated().any():
        duplicates = (
            frame.loc[
                frame["artifact_id"].duplicated(keep=False),
                "artifact_id",
            ]
            .astype(str)
            .unique()
            .tolist()
        )
        raise RuntimeError(f"{label} contains duplicate artifact IDs: {duplicates}")

    observed_ids = set(frame["artifact_id"].astype(str))
    missing_ids = sorted(set(required_ids) - observed_ids)

    if missing_ids:
        raise RuntimeError(
            f"{label} is missing required artifact IDs: {missing_ids}"
        )

    return frame


def manifest_path_lookup(frame):
    """Return artifact_id -> Path from an authoritative handoff manifest."""
    return {
        str(row.artifact_id): Path(str(row.path))
        for row in frame[["artifact_id", "path"]].itertuples(index=False)
    }


def artifact_inventory_row(
    artifact_id,
    path,
    role,
    consumer,
    ownership,
    purpose,
):
    """Create one compact manifest row from an existing file."""
    path = Path(path)
    require_file(path, artifact_id)

    if path.suffix.lower() == ".parquet":
        metadata = parquet_metadata(path)
        rows = metadata["rows"]
        columns = metadata["columns"]
        size_mb = metadata["size_mb"]
    else:
        rows = np.nan
        columns = np.nan
        size_mb = path.stat().st_size / (1024 ** 2)

    return {
        "artifact_id": artifact_id,
        "artifact_role": role,
        "downstream_consumer": consumer,
        "ownership": ownership,
        "path": str(path),
        "rows": rows,
        "columns": columns,
        "size_mb": size_mb,
        "purpose": purpose,
    }


def write_large_artifact_audit(path, audit_path, contract_id):
    """Freeze the cheap recovery evidence for one fully validated large file."""
    metadata = parquet_metadata(path)

    audit_df = pd.DataFrame([{
        "contract_id": contract_id,
        "path": str(path),
        **metadata,
        "status": "success",
    }])

    atomic_write_parquet(audit_df, audit_path)
    return audit_df


def large_artifact_matches_audit(path, audit_path, contract_id):
    """
    Cheaply determine whether a previously validated large artifact is reusable.

    We compare file metadata and the column-name signature rather than rescanning
    millions of rows during an ordinary notebook restart.
    """
    path = Path(path)
    audit_path = Path(audit_path)

    if not path.exists() or not audit_path.exists():
        return False

    audit_df = pd.read_parquet(audit_path)

    if len(audit_df) != 1:
        return False

    audit = audit_df.iloc[0]

    if str(audit["contract_id"]) != contract_id:
        return False

    current = parquet_metadata(path)

    checks = [
        int(audit["rows"]) == current["rows"],
        int(audit["columns"]) == current["columns"],
        int(audit["size_bytes"]) == current["size_bytes"],
        int(audit["modified_ns"]) == current["modified_ns"],
        str(audit["schema_signature"]) == current["schema_signature"],
        str(audit["status"]) == "success",
    ]

    return all(checks)


def summarize_showcase_surface(frame, group_columns):
    """
    Aggregate the frozen Showcase forecast surface at one requested grain.

    Native-unit errors are computed from the frozen champion-system prediction.
    Cross-target Relative MAE and benchmark skill retain the same definitions
    used elsewhere in the forecasting evaluation.
    """
    required_columns = [
        *group_columns,
        "actual",
        "champion_prediction",
        "benchmark_prediction",
        "champion_better_than_benchmark",
        "benchmark_loss",
        "severe_error",
        "system_row_error_index",
        "benchmark_advantage_index",
    ]
    require_columns(frame, required_columns, "Showcase forecast surface")

    work_df = frame[required_columns].copy()

    work_df["_signed_error"] = (
        work_df["champion_prediction"] - work_df["actual"]
    )
    work_df["_absolute_error"] = work_df["_signed_error"].abs()
    work_df["_squared_error"] = work_df["_signed_error"].pow(2)
    work_df["_benchmark_absolute_error"] = (
        work_df["benchmark_prediction"] - work_df["actual"]
    ).abs()
    work_df["_actual_absolute"] = work_df["actual"].abs()

    work_df["_champion_better"] = (
        work_df["champion_better_than_benchmark"].astype(float)
    )
    work_df["_benchmark_loss"] = work_df["benchmark_loss"].astype(float)
    work_df["_severe_error"] = work_df["severe_error"].astype(float)

    summary_df = (
        work_df.groupby(
            group_columns,
            dropna=False,
            observed=True,
        )
        .agg(
            forecast_rows=("actual", "size"),
            mae=("_absolute_error", "mean"),
            _mean_squared_error=("_squared_error", "mean"),
            bias=("_signed_error", "mean"),
            benchmark_mae=("_benchmark_absolute_error", "mean"),
            _champion_absolute_error_sum=("_absolute_error", "sum"),
            _actual_absolute_sum=("_actual_absolute", "sum"),
            champion_better_row_pct=("_champion_better", "mean"),
            benchmark_loss_pct=("_benchmark_loss", "mean"),
            severe_error_pct=("_severe_error", "mean"),
            mean_row_error_index=("system_row_error_index", "mean"),
            mean_benchmark_advantage_index=(
                "benchmark_advantage_index",
                "mean",
            ),
        )
        .reset_index()
    )

    summary_df["rmse"] = np.sqrt(summary_df["_mean_squared_error"])

    summary_df["benchmark_skill_pct"] = np.where(
        summary_df["benchmark_mae"].ne(0),
        100.0 * (1.0 - summary_df["mae"] / summary_df["benchmark_mae"]),
        np.nan,
    )

    summary_df["relative_mae_pct"] = np.where(
        summary_df["_actual_absolute_sum"].ne(0),
        100.0
        * summary_df["_champion_absolute_error_sum"]
        / summary_df["_actual_absolute_sum"],
        np.nan,
    )

    summary_df["champion_better_row_pct"] *= 100.0
    summary_df["benchmark_loss_pct"] *= 100.0
    summary_df["severe_error_pct"] *= 100.0

    return summary_df[
        [
            *group_columns,
            "forecast_rows",
            "mae",
            "rmse",
            "bias",
            "benchmark_mae",
            "benchmark_skill_pct",
            "relative_mae_pct",
            "champion_better_row_pct",
            "benchmark_loss_pct",
            "severe_error_pct",
            "mean_row_error_index",
            "mean_benchmark_advantage_index",
        ]
    ]

def qa_row(check, passed, detail):
    """Return one consistently shaped final-QA record."""
    return {
        "check": check,
        "passed": bool(passed),
        "detail": str(detail),
    }

## 4.7.1.1 Freeze the Chapter 5 Candidate Roster

The four forecasting branches have already completed their internal model-development work. Classical has frozen its final SARIMAX specification, Tree has frozen LightGBM, Neural has frozen its Lag-window MLP system, and the Transformer branch has frozen its PatchTST-inspired system.

Chapter 5 should **not repeat those searches**.

Instead, it receives one frozen candidate from each family for every Target × Horizon job. This creates a bounded **4 families × 5 targets × 3 horizons = 60-candidate roster**.

The modeling track recorded by each Chapter 4 candidate remains useful provenance: it tells us which ordinary forecasting history produced that family configuration. It does **not** mean Chapter 5 should reuse that ordinary track. Every counterfactual candidate will be re-evaluated later under the separate `pre_cp_reference` contract using only pre-policy training and validation data.

**4.7.1.1A — Validate and combine the four frozen family candidate registries.**

Before packaging anything, confirm that each family still contributes exactly the intended 15 Target × Horizon jobs and that its own completed QA still passes.

The common table keeps only the shared identity fields needed to organize the Chapter 5 mini-tournament. Exact architecture, feature, preprocessing, and output rules stay in the producer-owned family contracts rather than being copied into a misleading universal parameter schema.

In [9]:
# ---------------------------------------------------------------------
# 4.7.1.1A — Build the 60-row Chapter 5 candidate registry
# ---------------------------------------------------------------------

expected_jobs = expected_target_horizon_pairs()
candidate_parts = []
family_audit_records = []

for family in FAMILY_ORDER:
    spec = FAMILY_HANDOFFS[family]

    require_file(spec["registry"], f"{FAMILY_LABELS[family]} finalist registry")
    require_file(spec["qa"], f"{FAMILY_LABELS[family]} final QA")
    require_file(spec["manifest"], f"{FAMILY_LABELS[family]} final manifest")

    family_qa_passed, family_qa_df = qa_file_passes(spec["qa"])
    registry_df = pd.read_parquet(spec["registry"])
    normalized_df = normalize_finalist_registry(family, registry_df)

    observed_jobs = set(
        normalized_df[["metric", "horizon"]]
        .itertuples(index=False, name=None)
    )

    if len(normalized_df) != EXPECTED_JOB_COUNT:
        raise RuntimeError(
            f"{FAMILY_LABELS[family]} contributes {len(normalized_df)} jobs; "
            f"expected {EXPECTED_JOB_COUNT}."
        )

    if observed_jobs != expected_jobs:
        missing = sorted(expected_jobs - observed_jobs)
        extra = sorted(observed_jobs - expected_jobs)
        raise RuntimeError(
            f"{FAMILY_LABELS[family]} finalist job mismatch. "
            f"Missing={missing}; Extra={extra}"
        )

    if normalized_df.duplicated(["metric", "horizon"]).any():
        raise RuntimeError(
            f"{FAMILY_LABELS[family]} finalist registry is not unique by "
            "Target × Horizon."
        )

    if not family_qa_passed:
        raise RuntimeError(
            f"{FAMILY_LABELS[family]} upstream final QA no longer passes."
        )

    normalized_df["candidate_id"] = (
        normalized_df["family"]
        + "__"
        + normalized_df["metric"]
        + "__h"
        + normalized_df["horizon"].astype(str)
    )
    normalized_df["chapter5_modeling_track"] = "pre_cp_reference"
    normalized_df["chapter5_candidate_frozen"] = True
    normalized_df["chapter5_pre_cp_selected"] = False
    normalized_df["post_policy_evidence_allowed_for_chapter5_selection"] = False
    normalized_df["source_registry_path"] = str(spec["registry"])
    normalized_df["source_manifest_path"] = str(spec["manifest"])

    candidate_parts.append(normalized_df)

    family_audit_records.append({
        "family": family,
        "Family": FAMILY_LABELS[family],
        "jobs": len(normalized_df),
        "unique_configurations": normalized_df[
            "source_configuration_id"
        ].nunique(),
        "upstream_qa_rows": len(family_qa_df),
        "upstream_qa_passed": family_qa_passed,
        "source_manifest": str(spec["manifest"]),
    })


counterfactual_candidate_registry_df = (
    pd.concat(candidate_parts, ignore_index=True)
    .sort_values(["metric", "horizon", "family"])
    .reset_index(drop=True)
)

family_candidate_audit_df = pd.DataFrame(family_audit_records)

if len(counterfactual_candidate_registry_df) != EXPECTED_FAMILY_CANDIDATE_ROWS:
    raise RuntimeError(
        "The Chapter 5 candidate registry does not contain the expected "
        f"{EXPECTED_FAMILY_CANDIDATE_ROWS} rows."
    )

if counterfactual_candidate_registry_df.duplicated(
    ["family", "metric", "horizon"]
).any():
    raise RuntimeError(
        "The Chapter 5 candidate registry is not unique by "
        "Family × Target × Horizon."
    )

if counterfactual_candidate_registry_df["chapter5_pre_cp_selected"].any():
    raise RuntimeError(
        "4.7.1 must not pre-select a Chapter 5 counterfactual family."
    )

if WRITE_FINAL_OUTPUTS:
    atomic_write_parquet(
        counterfactual_candidate_registry_df,
        COUNTERFACTUAL_CANDIDATE_REGISTRY_PATH,
    )


display(
    family_candidate_audit_df[
        [
            "Family",
            "jobs",
            "unique_configurations",
            "upstream_qa_rows",
            "upstream_qa_passed",
        ]
    ].rename(
        columns={
            "jobs": "Jobs",
            "unique_configurations": "Frozen configurations",
            "upstream_qa_rows": "QA rows",
            "upstream_qa_passed": "QA passed",
        }
    )
)

print(
    "PASS — Chapter 5 candidate roster frozen.\n"
    f"  Families:   {counterfactual_candidate_registry_df['family'].nunique()}\n"
    f"  Jobs/family:{EXPECTED_JOB_COUNT}\n"
    f"  Candidates: {len(counterfactual_candidate_registry_df)}\n"
    f"  Selected:   "
    f"{int(counterfactual_candidate_registry_df['chapter5_pre_cp_selected'].sum())}"
)

,Family,Jobs,Frozen configurations,QA rows,QA passed
0,Classical,15,4,20,True
1,Tree,15,1,9,True
2,Neural,15,3,19,True
3,Transformer,15,2,21,True


PASS — Chapter 5 candidate roster frozen.
  Families:   4
  Jobs/family:15
  Candidates: 60
  Selected:   0


**Findings.** All four finalized forecasting families pass their producer-owned closeout checks and contribute a complete **15-job Target × Horizon roster**, producing exactly **60 frozen Chapter 5 candidates**—one candidate from each family for every Target × Horizon job.

The frozen configurations packaged here are the final model recipes selected within each family during Chapter 4:

| Model family    | Taxi trips                       | Taxi average speed              | FHVHV trips                             | FHVHV average speed                     | Subway ridership                            |
| --------------- | -------------------------------- | ------------------------------- | --------------------------------------- | --------------------------------------- | ------------------------------------------- |
| **Classical**   | MA(1) · raw<br>h=1, 2, 5         | AR(1) · raw<br>h=1, 2, 5        | AR(1) · raw: h=1<br>MA(1) · raw: h=2, 5 | AR(1) · raw: h=1, 2<br>MA(1) · raw: h=5 | AR(1) · log1p: h=1<br>MA(1) · log1p: h=2, 5 |
| **Tree**        | LightGBM · shared configuration  | LightGBM · shared configuration | LightGBM · shared configuration         | LightGBM · shared configuration         | LightGBM · shared configuration             |
| **Neural**      | Default MLP                      | Regularized MLP                 | Default MLP                             | Default MLP                             | Wider MLP                                   |
| **Transformer** | Higher capacity + regularization | Compact                         | Compact                                 | Higher capacity + regularization        | Compact                                     |

Repeated labels mean that multiple forecasting jobs reuse the same frozen recipe. Classical varies by both target and, in several cases, forecast horizon, producing its **4 model-recipe combinations** across the roster. Tree uses **1 shared LightGBM configuration**. Neural uses **3 MLP profiles**, while Transformer uses **2 capacity profiles**; all Transformer jobs retain the same two-week overlapping two-day input representation.

The original Chapter 4 training-history track is preserved separately as **provenance**, so it is intentionally not folded into the configuration labels above. Chapter 5 will apply these fixed recipes under the common `pre_cp_reference` experiment rather than inheriting their ordinary Full-History or Post-CP training data.

Most importantly, **0 of the 60 candidates are selected for Chapter 5**. This notebook has packaged the complete candidate universe without pre-deciding the Pre-CP Reference winner.


**4.7.1.1B — Publish the family-specific executable contract directory.**

The 60-row candidate table tells Chapter 5 **what may compete**. It does not contain enough information by itself to recreate four fundamentally different forecasting systems.

This companion manifest therefore tells Chapter 5 **where each candidate's real definition lives**.

Most of these assets describe the frozen model recipe: predictors, sequence construction, preprocessing, training policy, and forecast-output rules. The Neural branch also publishes the shared dense mobility-history state used by both Neural and Transformer to reconstruct literal historical sequence windows.

That history state contains canonical observed mobility values and observation masks—not fitted model weights. Chapter 5 may therefore reuse it as a historical source, but only through the observation sequence legally available at each forecast origin. Once the no-congestion-pricing counterfactual begins, observed post-policy mobility must not enter the recursive state.

In [10]:
# ---------------------------------------------------------------------
# 4.7.1.1B — Build the executable model-contract manifest
# ---------------------------------------------------------------------

model_contract_records = []

for family, contract_role, path, artifact_type, chapter5_use in MODEL_CONTRACT_SPECS:
    path = require_file(
        path,
        f"{FAMILY_LABELS[family]} · {contract_role}",
    )

    if artifact_type == "parquet":
        metadata = parquet_metadata(path)
        rows = metadata["rows"]
        columns = metadata["columns"]
        size_mb = metadata["size_mb"]
    else:
        rows = np.nan
        columns = np.nan
        size_mb = path.stat().st_size / (1024 ** 2)

    model_contract_records.append({
        "family": family,
        "Family": FAMILY_LABELS[family],
        "Representative": FAMILY_REPRESENTATIVES[family],
        "contract_role": contract_role,
        "artifact_type": artifact_type,
        "chapter5_use": chapter5_use,
        "path": str(path),
        "rows": rows,
        "columns": columns,
        "size_mb": size_mb,
    })


model_contract_manifest_df = (
    pd.DataFrame(model_contract_records)
    .sort_values(["family", "contract_role"])
    .reset_index(drop=True)
)

if WRITE_FINAL_OUTPUTS:
    atomic_write_parquet(
        model_contract_manifest_df,
        MODEL_CONTRACT_MANIFEST_PATH,
    )


display(
    model_contract_manifest_df.groupby(
        ["Family", "chapter5_use"],
        as_index=False,
        observed=True,
    ).agg(
        Artifacts=("contract_role", "size"),
        **{"Total size MB": ("size_mb", "sum")},
    )
)

print(
    "PASS — family-specific model contracts indexed.\n"
    f"  Contract references: {len(model_contract_manifest_df)}\n"
    f"  Output: {MODEL_CONTRACT_MANIFEST_PATH}"
)

,Family,chapter5_use,Artifacts,Total size MB
0,Classical,audit,1,0.003733
1,Classical,reuse,2,0.031839
2,Neural,audit,1,0.003263
3,Neural,reuse,6,16.840672
4,Transformer,audit,1,0.003259
5,Transformer,reuse,5,0.073079
6,Tree,audit,1,0.002937
7,Tree,reuse,3,0.048020


PASS — family-specific model contracts indexed.
  Contract references: 20
  Output: /datasets/_deepnote_work/pipeline_data/4.7.1.final_tables/counterfactual_model_contract_manifest.parquet


**Findings.** Chapter 5 now has a complete set of instructions and reusable inputs for reconstructing each frozen forecasting family under the Pre-CP Reference experiment.

These are **not additional model candidates**. They are separate contract files that describe different pieces of how each family's already-selected model should be rebuilt and run:

| Model family    | Frozen model recipe                   | Inputs / predictors        | Preprocessing / sequence setup                         | Training policy          | Forecast-output rules      | QA       |
| --------------- | ------------------------------------- | -------------------------- | ------------------------------------------------------ | ------------------------ | -------------------------- | -------- |
| **Classical**   | SARIMAX model contract                | Exogenous-feature contract | Included in SARIMAX specification                      | —                        | Included in model contract | Final QA |
| **Tree**        | Selected LightGBM registry            | Ordered predictor registry | —                                                      | —                        | Forecast-output contract   | Final QA |
| **Neural**      | MLP model-definition contract         | Neural input registry      | Sequence / preprocessing contract + shared history     | Training-policy evidence | Forecast-output contract   | Final QA |
| **Transformer** | Transformer model-definition contract | Transformer input registry | Context contract + shared Neural mobility-history state | Training-policy evidence | Forecast-output contract   | Final QA |

The number of files differs because the model families have different execution needs. A SARIMAX model can be specified relatively compactly, while the Neural and Transformer systems need explicit instructions for their sequence inputs, preprocessing, training behavior, and prediction format.

The shared mobility-history state is different from fitted model state. It preserves canonical observed mobility values and missingness masks so sequence models can reconstruct the exact history available at a particular forecast origin. Chapter 5 must still enforce the Pre-CP causal boundary when slicing that history and must build later recursive state from synthetic no-CP mobility rather than observed post-policy outcomes.

Chapter 5 will use these files as the **recipe book for rebuilding the same frozen model configurations on legal Pre-CP data**. They do not represent extra competitors, and no Chapter 4 fitted model weights are carried into the counterfactual experiment.

**4.7.1.1C — Preserve Chapter 4 champion and runner-up identities as provenance only.**

The ordinary forecasting tournament still matters for the Showcase and for understanding how Chapter 4 concluded. Those results should not disappear simply because Chapter 5 has a different causal-selection problem.

This cell therefore attaches the frozen Chapter 4 champion and pre-holdout runner-up identities to the candidate roster.

The flags are descriptive only. A family that won the post-policy Chapter 4 tournament receives **no automatic preference** in the Chapter 5 Pre-CP selection.

In [11]:
# ---------------------------------------------------------------------
# 4.7.1.1C — Preserve Chapter 4 champion / runner-up provenance
# ---------------------------------------------------------------------

required_461_ids = {
    "champion_registry",
    "champion_holdout_predictions",
    "pre_holdout_runner_up_registry",
    "champion_runner_up_holdout_rows",
}

handoff_461_df = read_handoff_manifest(
    FORECAST_EVALUATION_HANDOFF_461_PATH,
    required_461_ids,
    "4.6.1 evaluation handoff manifest",
)
handoff_461_paths = manifest_path_lookup(handoff_461_df)

champion_registry_df = pd.read_parquet(
    handoff_461_paths["champion_registry"]
)
runner_up_registry_df = pd.read_parquet(
    handoff_461_paths["pre_holdout_runner_up_registry"]
)

require_columns(
    champion_registry_df,
    [
        "metric",
        "horizon",
        "champion_family",
        "modeling_track",
        "configuration_id",
        "selection_basis",
        "champion_frozen",
        "holdout_used_for_selection",
    ],
    "4.6.1 champion registry",
)
require_columns(
    runner_up_registry_df,
    [
        "metric",
        "horizon",
        "champion_family",
        "runner_up_family",
        "selection_frozen_before_holdout",
        "holdout_used_for_selection",
    ],
    "4.6.1 pre-holdout runner-up registry",
)

chapter4_selection_provenance_df = (
    champion_registry_df.merge(
        runner_up_registry_df[
            [
                "metric",
                "horizon",
                "champion_family",
                "runner_up_family",
                "runner_up_basis",
                "validation_champion_mae",
                "validation_runner_up_mae",
                "validation_mae_advantage",
                "selection_frozen_before_holdout",
                "holdout_used_for_selection",
            ]
        ],
        on=["metric", "horizon", "champion_family"],
        how="left",
        validate="one_to_one",
        suffixes=("_champion", "_runner_up"),
    )
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

if len(chapter4_selection_provenance_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Chapter 4 selection provenance must contain exactly 15 jobs."
    )

if chapter4_selection_provenance_df[
    "holdout_used_for_selection_champion"
].astype(bool).any():
    raise RuntimeError(
        "A Chapter 4 champion now appears to use final-holdout evidence."
    )

champion_flags_df = chapter4_selection_provenance_df[
    ["metric", "horizon", "champion_family", "runner_up_family"]
].copy()

counterfactual_candidate_registry_df = (
    counterfactual_candidate_registry_df.merge(
        champion_flags_df,
        on=["metric", "horizon"],
        how="left",
        validate="many_to_one",
    )
)

counterfactual_candidate_registry_df["chapter4_champion"] = (
    counterfactual_candidate_registry_df["family"]
    .eq(counterfactual_candidate_registry_df["champion_family"])
)
counterfactual_candidate_registry_df["chapter4_runner_up"] = (
    counterfactual_candidate_registry_df["family"]
    .eq(counterfactual_candidate_registry_df["runner_up_family"])
)

counterfactual_candidate_registry_df[
    "chapter4_ranking_allowed_for_chapter5_selection"
] = False

if WRITE_FINAL_OUTPUTS:
    atomic_write_parquet(
        counterfactual_candidate_registry_df,
        COUNTERFACTUAL_CANDIDATE_REGISTRY_PATH,
    )
    atomic_write_parquet(
        chapter4_selection_provenance_df,
        CHAPTER4_SELECTION_PROVENANCE_PATH,
    )


display(
    counterfactual_candidate_registry_df.groupby(
        "Family",
        as_index=False,
        observed=True,
    ).agg(
        Candidates=("candidate_id", "size"),
        **{
            "Chapter 4 champions": ("chapter4_champion", "sum"),
            "Chapter 4 runner-ups": ("chapter4_runner_up", "sum"),
            "Chapter 5 selected": ("chapter5_pre_cp_selected", "sum"),
        },
    )
)

,Family,Candidates,Chapter 4 champions,Chapter 4 runner-ups,Chapter 5 selected
0,Classical,15,0,0,0
1,Neural,15,3,1,0
2,Transformer,15,4,10,0
3,Tree,15,8,4,0


**Findings.** The ordinary Chapter 4 tournament remains intact while the Chapter 5 candidate pool stays neutral.

### Section Summary

Section 4.7.1.1 converts four completed model-development branches into one bounded Chapter 5 candidate handoff.

The package contains **60 frozen Family × Target × Horizon candidates**, together with direct references to each family's real model, input, preprocessing, prediction, training-policy, QA, and manifest contracts. The ordinary Chapter 4 champion and runner-up labels remain attached as provenance, but **zero Chapter 5 winners have been selected**.

Chapter 5 can therefore compare the same four finalized forecasting ideas under a clean Pre-CP-only experiment without reopening architecture or hyperparameter development.

## 4.7.1.2 Package the Pre-CP Reference Contract

The model candidates are only one part of the counterfactual handoff. Chapter 5 also needs the exact forecasting experiment that defines what “Pre-CP Reference” means.

That experiment was already frozen upstream:

* model selection uses **2023 for initial training and calendar year 2024 for rolling validation**;
* model-selection targets end on **December 31, 2024**;
* **January 1–4, 2025** may be added only for the final refit after a model specification has been selected;
* the final legal observed state ends on **January 4, 2025 at sequence position 3,675**;
* recursive counterfactual application begins on **January 5, 2025 at sequence position 3,676**; and
* the primary observed-versus-counterfactual population is the previously frozen **1,029-sequence cross-regime intersection**, while the native Pre-CP modeling population remains **1,039 sequences**.

4.7.1 does not reinterpret those decisions. It validates and packages them so Chapter 5 can execute them directly.

**4.7.1.2A — Validate the Pre-CP time boundary and sequence populations.**

The first check protects the causal boundary itself.

The native Pre-CP population is retained for model fitting and validation. A second flag identifies the 1,029 sequences that also belong to the primary cross-regime population used later when observed post-policy mobility is compared with the no-policy forecast path.

No sequence is discarded merely because its observed mobility can equal zero. The support rule remains the upstream observed-history rule, not a retrospective forecast-error screen.

In [12]:
# ---------------------------------------------------------------------
# 4.7.1.2A — Validate and package Pre-CP time / sequence contracts
# ---------------------------------------------------------------------

pre_cp_refit_contract_df = pd.read_parquet(
    PRE_CP_REFIT_APPLICATION_CONTRACT_PATH
)
pre_cp_sequence_registry_df = pd.read_parquet(
    PRE_CP_SEQUENCE_REGISTRY_PATH
)
cross_regime_sequence_registry_df = pd.read_parquet(
    CROSS_REGIME_SEQUENCE_REGISTRY_PATH
)

require_columns(
    pre_cp_refit_contract_df,
    [
        "modeling_track",
        "model_selection_target_end_date",
        "refit_only_start_date",
        "final_refit_end_date",
        "final_refit_end_observation_sequence_id",
        "counterfactual_application_start_date",
        "counterfactual_application_start_observation_sequence_id",
        "counterfactual_application_end_date",
        "native_pre_cp_sequence_count",
        "primary_cross_regime_sequence_count",
    ],
    "Pre-CP refit/application contract",
)
require_columns(
    pre_cp_sequence_registry_df,
    [
        "forecast_sequence_id",
        "taxi_zone_id",
        "zone",
        "borough",
        "metric",
        "structurally_available",
        "eligibility_observed_share",
    ],
    "Pre-CP sequence registry",
)
require_columns(
    cross_regime_sequence_registry_df,
    ["forecast_sequence_id"],
    "Cross-regime sequence registry",
)

if len(pre_cp_refit_contract_df) != 1:
    raise RuntimeError(
        "The Pre-CP refit/application contract must contain exactly one row."
    )

contract = pre_cp_refit_contract_df.iloc[0]

date_checks = {
    "model_selection_target_end_date":
        PRE_CP_SELECTION_END_DATE,
    "refit_only_start_date":
        PRE_CP_REFIT_ONLY_START_DATE,
    "final_refit_end_date":
        PRE_CP_FINAL_REFIT_END_DATE,
    "counterfactual_application_start_date":
        COUNTERFACTUAL_START_DATE,
    "counterfactual_application_end_date":
        COUNTERFACTUAL_END_DATE,
}

for column, expected in date_checks.items():
    observed = pd.Timestamp(contract[column]).normalize()

    if observed != expected:
        raise RuntimeError(
            f"{column} changed from {expected.date()} to {observed.date()}."
        )

if int(contract["final_refit_end_observation_sequence_id"]) != (
    PRE_CP_FINAL_REFIT_SEQUENCE_ID
):
    raise RuntimeError("The final Pre-CP refit sequence boundary changed.")

if int(
    contract["counterfactual_application_start_observation_sequence_id"]
) != COUNTERFACTUAL_START_SEQUENCE_ID:
    raise RuntimeError("The counterfactual start sequence boundary changed.")

if len(pre_cp_sequence_registry_df) != EXPECTED_PRE_CP_NATIVE_SEQUENCES:
    raise RuntimeError(
        "Native Pre-CP sequence population changed: "
        f"{len(pre_cp_sequence_registry_df):,}."
    )

if len(cross_regime_sequence_registry_df) != (
    EXPECTED_PRIMARY_CROSS_REGIME_SEQUENCES
):
    raise RuntimeError(
        "Primary cross-regime sequence population changed: "
        f"{len(cross_regime_sequence_registry_df):,}."
    )


cross_regime_ids = set(
    cross_regime_sequence_registry_df["forecast_sequence_id"]
)
native_pre_cp_ids = set(
    pre_cp_sequence_registry_df["forecast_sequence_id"]
)

if not cross_regime_ids.issubset(native_pre_cp_ids):
    raise RuntimeError(
        "The primary cross-regime population is no longer a subset of "
        "the native Pre-CP sequence population."
    )


pre_cp_sequence_support_registry_df = (
    pre_cp_sequence_registry_df.copy()
)

pre_cp_sequence_support_registry_df["primary_cross_regime"] = (
    pre_cp_sequence_support_registry_df["forecast_sequence_id"]
    .isin(cross_regime_ids)
)

pre_cp_sequence_support_registry_df["support_contract_source"] = "4.1.1/4.1.2"
pre_cp_sequence_support_registry_df[
    "genuine_observed_zero_is_valid"
] = True
pre_cp_sequence_support_registry_df[
    "holdout_error_used_for_support"
] = False

if WRITE_FINAL_OUTPUTS:
    atomic_write_parquet(
        pre_cp_sequence_support_registry_df,
        PRE_CP_SEQUENCE_SUPPORT_REGISTRY_PATH,
    )


display(
    pd.DataFrame([
        {
            "Population": "Native Pre-CP Reference",
            "Forecast Sequences": len(pre_cp_sequence_registry_df),
            "Role": "Chapter 5 fitting and Pre-CP validation",
        },
        {
            "Population": "Primary cross-regime intersection",
            "Forecast Sequences": len(cross_regime_sequence_registry_df),
            "Role": "Observed vs no-CP comparison",
        },
    ])
)

print(
    "PASS — Pre-CP causal boundary preserved.\n"
    f"  Selection ends:    {PRE_CP_SELECTION_END_DATE.date()}\n"
    f"  Final refit ends:  {PRE_CP_FINAL_REFIT_END_DATE.date()} "
    f"(sequence {PRE_CP_FINAL_REFIT_SEQUENCE_ID})\n"
    f"  Application starts:{COUNTERFACTUAL_START_DATE.date()} "
    f"(sequence {COUNTERFACTUAL_START_SEQUENCE_ID})"
)

,Population,Forecast Sequences,Role
0,Native Pre-CP Reference,1039,Chapter 5 fitting and Pre-CP validation
1,Primary cross-regime intersection,1029,Observed vs no-CP comparison


PASS — Pre-CP causal boundary preserved.
  Selection ends:    2024-12-31
  Final refit ends:  2025-01-04 (sequence 3675)
  Application starts:2025-01-05 (sequence 3676)


**Findings.** The complete Pre-CP Reference boundary validates exactly as designed.

The native Pre-CP modeling population contains **1,039 Forecast Sequences**, while the primary cross-regime comparison population contains **1,029**. The latter remains a strict subset of the native modeling population rather than replacing it.

Model selection still ends on **December 31, 2024**. January 1–4, 2025 remain available only for the final refit, with the last legal observed state at sequence **3,675**. Counterfactual application begins on **January 5, 2025 at sequence 3,676**.

The packaged support contract also preserves genuine observed zero activity as valid data and does not introduce any holdout-error-based exclusion rule.

**4.7.1.2B — Freeze the Pre-CP benchmark and rolling-validation experiment.**

Chapter 5 needs a common simple benchmark when it compares the four frozen families on 2024 Pre-CP validation.

That benchmark has already been selected upstream. The advanced-model benchmark registry contains one benchmark for every Modeling Track × Target × Horizon combination, including all 15 Pre-CP Reference jobs.

4.7.1 therefore subsets and validates that existing registry rather than running another baseline tournament.

In [13]:
# ---------------------------------------------------------------------
# 4.7.1.2B — Package the 15-job Pre-CP benchmark registry
# ---------------------------------------------------------------------

advanced_benchmark_registry_df = pd.read_parquet(
    ADVANCED_BENCHMARK_REGISTRY_PATH
)

require_columns(
    advanced_benchmark_registry_df,
    [
        "modeling_track",
        "metric",
        "horizon",
        "baseline_id",
        "baseline_label",
        "selection_metric",
        "selection_stage",
    ],
    "Advanced-model benchmark registry",
)

pre_cp_benchmark_registry_df = (
    advanced_benchmark_registry_df.loc[
        advanced_benchmark_registry_df["modeling_track"]
        .eq("pre_cp_reference")
    ]
    .copy()
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

if len(pre_cp_benchmark_registry_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "The Pre-CP benchmark registry must contain exactly 15 jobs."
    )

if pre_cp_benchmark_registry_df.duplicated(
    ["modeling_track", "metric", "horizon"]
).any():
    raise RuntimeError(
        "The Pre-CP benchmark registry is not unique by "
        "Track × Target × Horizon."
    )

observed_jobs = set(
    pre_cp_benchmark_registry_df[
        ["metric", "horizon"]
    ].itertuples(index=False, name=None)
)

if observed_jobs != expected_target_horizon_pairs():
    raise RuntimeError(
        "The Pre-CP benchmark registry no longer covers all 15 jobs."
    )

if not pre_cp_benchmark_registry_df[
    "selection_stage"
].eq("rolling_validation").all():
    raise RuntimeError(
        "A Pre-CP benchmark is no longer frozen from rolling validation."
    )

if WRITE_FINAL_OUTPUTS:
    atomic_write_parquet(
        pre_cp_benchmark_registry_df,
        PRE_CP_BENCHMARK_REGISTRY_PATH,
    )


display(
    pre_cp_benchmark_registry_df[
        [
            "metric",
            "horizon",
            "baseline_label",
            "selection_metric",
            "selection_stage",
        ]
    ].assign(
        Target=lambda frame: frame["metric"].map(TARGET_LABELS)
    )[
        [
            "Target",
            "horizon",
            "baseline_label",
            "selection_metric",
            "selection_stage",
        ]
    ]
)

,Target,horizon,baseline_label,selection_metric,selection_stage
0,FHVHV speed,1,Previous week · same Daypart,mae,rolling_validation
1,FHVHV speed,2,Previous week · same Daypart,mae,rolling_validation
2,FHVHV speed,5,Previous week · same Daypart,mae,rolling_validation
3,FHVHV trips,1,Previous week · same Daypart,mae,rolling_validation
4,FHVHV trips,2,Previous week · same Daypart,mae,rolling_validation
5,FHVHV trips,5,Previous week · same Daypart,mae,rolling_validation
6,Subway ridership,1,Previous week · same Daypart,mae,rolling_validation
7,Subway ridership,2,Previous week · same Daypart,mae,rolling_validation
8,Subway ridership,5,Previous week · same Daypart,mae,rolling_validation
9,Taxi speed,1,Recent 7-day Daypart mean,mae,rolling_validation


**Findings.** The Pre-CP Reference benchmark is complete for all **15 Target × Horizon jobs** and was selected entirely from rolling-validation evidence.

The frozen benchmark is **Previous week · same Daypart** for all Taxi trips, FHVHV trips, FHVHV speed, and Subway ridership jobs. Taxi speed is the one exception: all three Taxi-speed horizons use the **Recent 7-day Daypart mean**.

Chapter 5 therefore enters its family mini-tournament with one already-frozen simple reference for every job; no new baseline selection is required.

**4.7.1.2C — Publish the complete Pre-CP Reference handoff without duplicating upstream data.**

Chapter 5 needs one directory for the full Pre-CP forecasting experiment, but the authoritative inputs remain owned by the notebooks that created them.

The large historical forecasting matrix, the final-refit-only training supplement, and the counterfactual application template therefore remain at their 4.2.3 locations. The historical matrix carries the 2023/2024 family-selection experiment, the supplement carries the additional supervised January 1–4 history used only for final refitting, and the application template defines the January 5 onward recursive no-CP forecast surface.

The same ownership rule applies to the row-level selected benchmark surface published by 4.2.4: Chapter 5 needs those frozen benchmark predictions for aligned validation comparison and fallback, but 4.7.1 does not need to copy them.

This cell therefore creates a **pointer manifest**. Small 4.7.1 registries are owned here, while historical, final-refit, feature, recursive-runtime, benchmark-prediction, and application assets remain at their authoritative upstream locations.

This is the directory Chapter 5 should use instead of searching across prior notebooks.

In [14]:
# ---------------------------------------------------------------------
# 4.7.1.2C — Publish the complete Pre-CP Reference handoff manifest
# ---------------------------------------------------------------------

pre_cp_handoff_specs = [
    (
        "counterfactual_candidate_registry",
        COUNTERFACTUAL_CANDIDATE_REGISTRY_PATH,
        "Chapter 5 candidate roster",
        "Chapter 5",
        "4.7.1_owned",
        "Four frozen family candidates for every Target × Horizon.",
    ),
    (
        "counterfactual_model_contract_manifest",
        MODEL_CONTRACT_MANIFEST_PATH,
        "Model execution directory",
        "Chapter 5",
        "4.7.1_owned",
        "Pointers to authoritative family-specific executable contracts.",
    ),
    (
        "pre_cp_sequence_support_registry",
        PRE_CP_SEQUENCE_SUPPORT_REGISTRY_PATH,
        "Forecast support and cohort membership",
        "Chapter 5",
        "4.7.1_owned",
        "Native Pre-CP sequence support plus primary cross-regime membership.",
    ),
    (
        "pre_cp_benchmark_registry",
        PRE_CP_BENCHMARK_REGISTRY_PATH,
        "Benchmark contract",
        "Chapter 5",
        "4.7.1_owned",
        "Frozen benchmark for each Pre-CP Target × Horizon job.",
    ),
    (
        "pre_cp_selected_benchmark_predictions",
        SELECTED_PRE_CP_BENCHMARK_PREDICTIONS_PATH,
        "Row-level selected benchmark surface",
        "Chapter 5",
        "upstream_reference",
        (
            "Frozen selected benchmark prediction, observed target, and "
            "support context for each legal 2024 Pre-CP validation row."
        ),
    ),
    (
        "rolling_origin_registry",
        ROLLING_ORIGIN_PATH,
        "Pre-CP model-selection schedule",
        "Chapter 5",
        "upstream_reference",
        "Authoritative 2023 training / 2024 rolling-validation schedule.",
    ),
    (
        "forecasting_track_contract",
        TRACK_CONTRACT_PATH,
        "Modeling-track semantics",
        "Chapter 5",
        "upstream_reference",
        "Frozen definitions of all three forecasting tracks.",
    ),
    (
        "pre_cp_refit_application_contract",
        PRE_CP_REFIT_APPLICATION_CONTRACT_PATH,
        "Final refit and application boundary",
        "Chapter 5",
        "upstream_reference",
        "Executable Dec. 31 / Jan. 4 / Jan. 5 Pre-CP boundary contract.",
    ),
    (
        "pre_cp_native_sequence_registry",
        PRE_CP_SEQUENCE_REGISTRY_PATH,
        "Native Pre-CP population",
        "Chapter 5",
        "upstream_reference",
        "1,039 Forecast Sequences eligible for Pre-CP model selection.",
    ),
    (
        "cross_regime_sequence_registry",
        CROSS_REGIME_SEQUENCE_REGISTRY_PATH,
        "Primary causal-comparison population",
        "Chapter 5",
        "upstream_reference",
        "1,029-sequence observed-versus-counterfactual intersection.",
    ),
    (
        "forecast_target_context",
        FORECAST_TARGET_CONTEXT_PATH,
        "Horizon-specific target calendar",
        "Chapter 5",
        "upstream_reference",
        "Target date, Daypart, temporal bucket, and horizon alignment.",
    ),
    (
        "pre_cp_counterfactual_calendar",
        PRE_CP_COUNTERFACTUAL_CALENDAR_PATH,
        "Recursive application calendar",
        "Chapter 5",
        "upstream_reference",
        "Future-known origin × horizon calendar after the Pre-CP freeze.",
    ),
    (
        "pre_cp_frozen_weekly_state",
        PRE_CP_COUNTERFACTUAL_WEEKLY_STATE_PATH,
        "Frozen seasonal state",
        "Chapter 5",
        "upstream_reference",
        "Final legal weekly seasonal state carried beyond Jan. 4, 2025.",
    ),
    (
        "historical_weekly_stl_selection_registry",
        WEEKLY_STL_SELECTION_REGISTRY_PATH,
        "Historical weekly STL selection registry",
        "Chapter 5",
        "upstream_reference",
        "Historical weekly STL selections available at each forecast origin.",
    ),
    (
        "historical_weekly_stl_template_interval_registry",
        WEEKLY_STL_TEMPLATE_INTERVAL_REGISTRY_PATH,
        "Historical weekly STL template intervals",
        "Chapter 5",
        "upstream_reference",
        "Historical weekly STL template intervals available at each forecast origin.",
    ),
    (
        "pre_cp_recursive_feature_contract",
        PRE_CP_RECURSIVE_FEATURE_CONTRACT_PATH,
        "Recursive mobility-feature rules",
        "Chapter 5",
        "upstream_reference",
        "Defines which mobility-history features must be rebuilt recursively.",
    ),
    (
        "pre_cp_context_runtime_contract",
        PRE_CP_CONTEXT_RUNTIME_CONTRACT_PATH,
        "Context availability rules",
        "Chapter 5",
        "upstream_reference",
        "Defines post-freeze multimodal/spatial context behavior.",
    ),
    (
        "complete_forecast_feature_catalog",
        COMPLETE_FEATURE_CATALOG_PATH,
        "Shared predictor definitions",
        "Chapter 5",
        "upstream_reference",
        "Authoritative cross-family forecasting feature catalog.",
    ),
        (
        "pre_cp_historical_forecasting_matrix",
        PRE_CP_FORECASTING_MATRIX_PATH,
        "Historical model-selection matrix",
        "Chapter 5",
        "upstream_reference",
        (
            "2023 initial-training and 2024 family-selection matrix "
            "for the Pre-CP Reference."
        ),
    ),
    (
        "pre_cp_final_refit_training_supplement",
        PRE_CP_FINAL_REFIT_SUPPLEMENT_PATH,
        "Final-refit-only supervised supplement",
        "Chapter 5",
        "upstream_reference",
        (
            "Additional supervised Pre-CP rows for January 1–4, 2025, "
            "used only after the Chapter 5 family roster is frozen."
        ),
    ),
    (
        "pre_cp_final_runtime_contract",
        PRE_CP_FINAL_RUNTIME_CONTRACT_PATH,
        "Post-freeze feature behavior",
        "Chapter 5",
        "upstream_reference",
        "Action required for every shared feature after the Pre-CP freeze.",
    ),
    (
        "pre_cp_counterfactual_application_template",
        PRE_CP_COUNTERFACTUAL_TEMPLATE_PATH,
        "No-CP recursive application scaffold",
        "Chapter 5",
        "upstream_reference",
        "Runtime-aware post-policy template; contains no observed post-CP target.",
    ),
    (
        "final_forecasting_matrix_qa",
        FINAL_FORECASTING_MATRIX_QA_PATH,
        "Feature/matrix leakage QA",
        "Chapter 5",
        "upstream_reference",
        "Completed cross-family leakage and handoff checks.",
    ),
    (
        "forecasting_matrix_manifest",
        FORECASTING_MATRIX_MANIFEST_PATH,
        "Feature/matrix directory",
        "Chapter 5",
        "upstream_reference",
        "Authoritative inventory for the completed forecasting matrices.",
    ),
]

pre_cp_handoff_manifest_df = pd.DataFrame([
    artifact_inventory_row(
        artifact_id,
        path,
        role,
        consumer,
        ownership,
        purpose,
    )
    for artifact_id, path, role, consumer, ownership, purpose
    in pre_cp_handoff_specs
])

if len(pre_cp_handoff_manifest_df) != 24:
    raise RuntimeError(
        "Pre-CP handoff manifest must contain exactly 24 artifacts; "
        f"found {len(pre_cp_handoff_manifest_df)}."
    )

if pre_cp_handoff_manifest_df["artifact_id"].duplicated().any():
    raise RuntimeError(
        "Pre-CP handoff manifest contains duplicate artifact IDs."
    )

if WRITE_FINAL_OUTPUTS:
    atomic_write_parquet(
        pre_cp_handoff_manifest_df,
        PRE_CP_HANDOFF_MANIFEST_PATH,
    )


display(
    pre_cp_handoff_manifest_df[
        [
            "artifact_id",
            "artifact_role",
            "ownership",
            "rows",
            "size_mb",
        ]
    ].rename(
        columns={
            "artifact_id": "Artifact",
            "artifact_role": "Role",
            "ownership": "Ownership",
            "rows": "Rows",
            "size_mb": "Size MB",
        }
    ).round(3)
)

,Artifact,Role,Ownership,Rows,Size MB
0,counterfactual_candidate_registry,Chapter 5 candidate roster,4.7.1_owned,60,0.109
1,counterfactual_model_contract_manifest,Model execution directory,4.7.1_owned,20,0.008
2,pre_cp_sequence_support_registry,Forecast support and cohort membership,4.7.1_owned,1039,0.021
3,pre_cp_benchmark_registry,Benchmark contract,4.7.1_owned,15,0.016
4,pre_cp_selected_benchmark_predictions,Row-level selected benchmark surface,upstream_reference,5680584,42.082
5,rolling_origin_registry,Pre-CP model-selection schedule,upstream_reference,35496,0.243
6,forecasting_track_contract,Modeling-track semantics,upstream_reference,3,0.013
7,pre_cp_refit_application_contract,Final refit and application boundary,upstream_reference,1,0.009
8,pre_cp_native_sequence_registry,Native Pre-CP population,upstream_reference,1039,0.023
9,cross_regime_sequence_registry,Primary causal-comparison population,upstream_reference,1029,0.018


**Findings.** The complete Pre-CP Reference handoff is available through one compact directory without copying its largest upstream data files.

Three supervised/application assets define the main lifecycle:

| Asset | What it is used for |
| --- | --- |
| **Historical Pre-CP forecasting matrix** — 11,384,323 rows | Historical training and 2024 validation data used to compare the frozen model families. |
| **Final-refit training supplement** — 62,340 rows | Additional observed January 1–4, 2025 training rows used only after the Chapter 5 family roster is frozen. |
| **Counterfactual forecast template** — 7,023,640 rows | Forecast rows that Chapter 5 fills from **January 5, 2025 onward** when it simulates mobility under a no-congestion-pricing scenario. |

The counterfactual template is best thought of as an **empty forecast skeleton**. It already identifies where forecasts must be produced and carries the calendar, geography, horizon, and other runtime context needed for those rows, but it does **not** contain the no-CP mobility predictions.

Chapter 5 also receives the frozen **row-level Pre-CP benchmark surface** from 4.2.4. The 15-job benchmark registry identifies **which simple benchmark applies to each Target × Horizon**, while the row-level surface provides the corresponding benchmark prediction for each legal 2024 validation row, together with the support information needed for fair comparison and fallback.

After Chapter 5 selects the Pre-CP Reference family for each Target × Horizon, the selected models use the historical matrix plus the January 1–4 supplement for their final refit through January 4. From January 5 onward, the counterfactual template is advanced recursively from prior **predicted no-CP mobility**, not from actual post-policy observations.

4.7.1 therefore packages the small candidate, support, benchmark, and model-definition registries needed downstream while pointing Chapter 5 to the authoritative historical, final-refit, benchmark, and recursive-application assets.

### Section Summary

The Pre-CP Reference contract is packaged as an executable downstream handoff rather than a collection of notebook-specific assumptions.

Chapter 5 receives the frozen 2024 validation experiment, the separate January 1–4 final-refit training supplement, the January 4 final-refit boundary, the January 5 application boundary, the 1,039 native Pre-CP sequences, the 1,029 primary cross-regime population, the frozen benchmark, the leakage-safe historical forecasting matrix, and the recursive feature/runtime/application contracts.

No counterfactual model has been selected or fit in 4.7.1.

## 4.7.1.3 Package the Showcase Forecast Surfaces

The Showcase needs two different views of the frozen Chapter 4 forecasting system.

The **final-holdout surface** is the clean evaluation layer. It contains the benchmark-supported January–March 2026 records used to judge how the frozen system performed on unseen data and carries the detailed reliability context from 4.6.2.

The **longer forecast-history surface** serves a different purpose. It follows the same 15 frozen champion jobs back through ordinary rolling validation so readers can explore a longer timeline without treating those earlier rows as untouched holdout evidence.

For both surfaces, the forecast of record follows the same production rule: use the advanced champion when it is available and the frozen previous-week benchmark as fallback otherwise. The evaluation period is kept explicit so downstream pages cannot blur ordinary validation and final holdout.

**4.7.1.3A — Resolve and validate the authoritative 4.6.1 and 4.6.2 row-level sources.**

The producer manifests remain the source of truth for physical locations. This avoids assuming that every important artifact lives in a `final_tables` directory—4.6.1 legitimately keeps some frozen provenance in its intermediate directory.

Before the large merge begins, check only metadata and required schemas. Full row-level semantic validation happens once after the new Showcase artifact is created.

In [15]:
# ---------------------------------------------------------------------
# 4.7.1.3A — Resolve the authoritative Showcase inputs
# ---------------------------------------------------------------------

required_462_ids = {
    "record_diagnostics",
    "reliability_guidance",
    "job_reliability_summary",
    "failure_mechanism_summary",
    "failure_case_studies",
    "failure_case_context",
    "model_interpretation_coverage",
    "tree_training_history_sensitivity",
}

handoff_462_df = read_handoff_manifest(
    FORECAST_EVALUATION_HANDOFF_462_PATH,
    required_462_ids,
    "4.6.2 evaluation handoff manifest",
)
handoff_462_paths = manifest_path_lookup(handoff_462_df)

runner_up_holdout_rows_path = handoff_461_paths[
    "champion_runner_up_holdout_rows"
]
record_diagnostics_path = handoff_462_paths[
    "record_diagnostics"
]

require_parquet_columns(
    runner_up_holdout_rows_path,
    [
        *COMMON_FORECAST_KEY,
        "champion_family",
        "runner_up_family",
        "target_date",
        "actual",
        "benchmark_prediction",
        "champion_system_prediction",
        "runner_up_system_prediction",
        "champion_model_available",
        "runner_up_model_available",
        "runner_up_used_benchmark_fallback",
    ],
    "4.6.1 champion / runner-up holdout rows",
)

require_parquet_columns(
    record_diagnostics_path,
    [
        *PHYSICAL_FORECAST_KEY,
        "champion_family",
        "zone",
        "borough",
        "target_date",
        "target_temporal_bucket",
        "actual",
        "prediction_final",
        "benchmark_prediction",
        "row_error_index",
        "benchmark_loss",
        "severe_error",
        "failure_combination",
        "low_zero_demand",
        "unstable_conditions",
        "shared_shock",
    ],
    "4.6.2 record diagnostics",
)

runner_metadata = parquet_metadata(runner_up_holdout_rows_path)
diagnostic_metadata = parquet_metadata(record_diagnostics_path)

if runner_metadata["rows"] != EXPECTED_SHOWCASE_ROWS:
    raise RuntimeError(
        "4.6.1 benchmark-comparable row count changed: "
        f"{runner_metadata['rows']:,}."
    )

if diagnostic_metadata["rows"] != EXPECTED_SHOWCASE_ROWS:
    raise RuntimeError(
        "4.6.2 record-diagnostic row count changed: "
        f"{diagnostic_metadata['rows']:,}."
    )

display(
    pd.DataFrame([
        {
            "Source": "4.6.1 champion / runner-up rows",
            "Rows": runner_metadata["rows"],
            "Columns": runner_metadata["columns"],
            "Role": "Forecasts of record",
        },
        {
            "Source": "4.6.2 record diagnostics",
            "Rows": diagnostic_metadata["rows"],
            "Columns": diagnostic_metadata["columns"],
            "Role": "Explanatory context",
        },
    ])
)

print(
    "PASS — benchmark-comparable Showcase inputs are aligned by size.\n"
    f"  Frozen rows: {EXPECTED_SHOWCASE_ROWS:,}"
)

,Source,Rows,Columns,Role
0,4.6.1 champion / runner-up rows,1421436,20,Forecasts of record
1,4.6.2 record diagnostics,1421436,26,Explanatory context


PASS — benchmark-comparable Showcase inputs are aligned by size.
  Frozen rows: 1,421,436


**Findings.** The two authoritative row-level Showcase inputs align exactly before packaging.

The 4.6.1 champion/runner-up forecast surface and the 4.6.2 diagnostic surface each contain **1,421,436 records**. The former remains the source of forecast values; the latter supplies explanatory context only.

This confirms that the Showcase handoff can enrich the frozen forecast population without changing its denominator.

**4.7.1.3B — Build or recover the row-level Showcase forecast surface.**

This step combines the frozen 4.6.1 forecast outputs with the 4.6.2 diagnostic context using **Pandas DataFrame operations**. Only the columns needed for the merge and downstream summaries are loaded, which keeps the operation manageable without introducing a separate SQL workflow.

For every forecast record, the output preserves:

* observed mobility;
* the frozen Chapter 4 champion-system forecast;
* the frozen weekly benchmark;
* the frozen runner-up system forecast;
* native-unit champion, benchmark, and runner-up errors;
* scale-aware error measures that allow forecasts from different mobility targets to be compared more fairly;
* the 4.6.2 diagnostic flags; and
* a physical-geography flag identifying whether the record belongs to a valid Taxi Zone that can be used in zone-level maps and geographic analyses.

Rows associated with `Unknown` or nonphysical Taxi Zones remain in the full analytical dataset. They are excluded only from analyses that require a valid geographic location.

`row_error_index` remains the upstream scale-aware **miss-severity** measure. A second within-job index measures **benchmark advantage**, allowing especially strong forecasts to be ranked across targets without treating trips, riders, and miles per hour as though they share the same native scale.


In [16]:
# ---------------------------------------------------------------------
# 4.7.1.3B — Build or recover the Showcase forecast record surface
# ---------------------------------------------------------------------

showcase_surface_reusable = large_artifact_matches_audit(
    SHOWCASE_FORECAST_RECORD_SURFACE_PATH,
    SHOWCASE_FORECAST_RECORD_AUDIT_PATH,
    SHOWCASE_SURFACE_CONTRACT_ID,
)

if (
    RUN_SHOWCASE_SURFACE_BUILD
    and (
        REBUILD_SHOWCASE_SURFACE
        or not showcase_surface_reusable
    )
):
    runner_columns = [
        *COMMON_FORECAST_KEY,
        "champion_family",
        "runner_up_family",
        "target_date",
        "actual",
        "benchmark_prediction",
        "champion_system_prediction",
        "runner_up_system_prediction",
        "champion_model_available",
        "runner_up_model_available",
        "runner_up_used_benchmark_fallback",
    ]

    diagnostic_columns = [
        *PHYSICAL_FORECAST_KEY,
        "champion_family",
        "zone",
        "borough",
        "target_temporal_bucket",
        "actual",
        "prediction_final",
        "benchmark_prediction",
        "row_error_index",
        "benchmark_loss",
        "severe_error",
        "failure_combination",
        "low_zero_demand",
        "unstable_conditions",
        "shared_shock",
    ]

    runner_df = pd.read_parquet(
        runner_up_holdout_rows_path,
        columns=runner_columns,
    )

    diagnostic_df = (
        pd.read_parquet(
            record_diagnostics_path,
            columns=diagnostic_columns,
        )
        .rename(
            columns={
                "champion_family": "diagnostic_champion_family",
                "actual": "diagnostic_actual",
                "benchmark_prediction":
                    "diagnostic_benchmark_prediction",
                "prediction_final":
                    "diagnostic_advanced_prediction",
                "row_error_index":
                    "diagnostic_row_error_index",
            }
        )
    )

    if runner_df.duplicated(COMMON_FORECAST_KEY).any():
        raise RuntimeError(
            "4.6.1 champion / runner-up rows are not unique by "
            "the frozen forecast key."
        )

    if diagnostic_df.duplicated(COMMON_FORECAST_KEY).any():
        raise RuntimeError(
            "4.6.2 record diagnostics are not unique by "
            "the frozen forecast key."
        )

    showcase_surface_df = runner_df.merge(
        diagnostic_df,
        on=COMMON_FORECAST_KEY,
        how="inner",
        validate="one_to_one",
    )

    if len(showcase_surface_df) != EXPECTED_SHOWCASE_ROWS:
        raise RuntimeError(
            "The 4.6.1 / 4.6.2 Showcase merge changed the frozen "
            f"row count: {len(showcase_surface_df):,}."
        )

    actual_matches = np.isclose(
        showcase_surface_df["actual"],
        showcase_surface_df["diagnostic_actual"],
        rtol=0.0,
        atol=CROSS_ARTIFACT_ATOL,
        equal_nan=True,
    )
    benchmark_matches = np.isclose(
        showcase_surface_df["benchmark_prediction"],
        showcase_surface_df["diagnostic_benchmark_prediction"],
        rtol=0.0,
        atol=CROSS_ARTIFACT_ATOL,
        equal_nan=True,
    )
    family_matches = showcase_surface_df["champion_family"].eq(
        showcase_surface_df["diagnostic_champion_family"]
    )

    if not (
        actual_matches.all()
        and benchmark_matches.all()
        and family_matches.all()
    ):
        raise RuntimeError(
            "4.6.1 and 4.6.2 disagree on the observed outcome, benchmark, "
            "or champion family for at least one frozen Showcase row."
        )

    showcase_surface_df = (
        showcase_surface_df.rename(
            columns={
                "champion_system_prediction": "champion_prediction",
                "runner_up_system_prediction": "runner_up_prediction",
            }
        )
        .drop(
            columns=[
                "diagnostic_actual",
                "diagnostic_benchmark_prediction",
                "diagnostic_champion_family",
            ]
        )
    )

    showcase_surface_df["signed_error"] = (
        showcase_surface_df["champion_prediction"]
        - showcase_surface_df["actual"]
    )
    showcase_surface_df["absolute_error"] = (
        showcase_surface_df["signed_error"].abs()
    )
    showcase_surface_df["squared_error"] = (
        showcase_surface_df["signed_error"].pow(2)
    )

    showcase_surface_df["benchmark_absolute_error"] = (
        showcase_surface_df["benchmark_prediction"]
        - showcase_surface_df["actual"]
    ).abs()

    showcase_surface_df["runner_up_absolute_error"] = (
        showcase_surface_df["runner_up_prediction"]
        - showcase_surface_df["actual"]
    ).abs()

    showcase_surface_df["champion_advantage_abs"] = (
        showcase_surface_df["benchmark_absolute_error"]
        - showcase_surface_df["absolute_error"]
    )

    showcase_surface_df["champion_better_than_benchmark"] = (
        showcase_surface_df["absolute_error"]
        < showcase_surface_df["benchmark_absolute_error"]
    )

    showcase_surface_df["job_champion_mae"] = (
        showcase_surface_df.groupby(
            ["metric", "horizon"],
            observed=True,
        )["absolute_error"]
        .transform("mean")
    )

    positive_job_mae = showcase_surface_df["job_champion_mae"].gt(0)

    showcase_surface_df["system_row_error_index"] = np.where(
        positive_job_mae,
        100.0
        * showcase_surface_df["absolute_error"]
        / showcase_surface_df["job_champion_mae"],
        np.nan,
    )

    showcase_surface_df["benchmark_advantage_index"] = np.where(
        positive_job_mae,
        100.0
        * showcase_surface_df["champion_advantage_abs"]
        / showcase_surface_df["job_champion_mae"],
        np.nan,
    )

    showcase_surface_df["reader_facing_zone"] = (
        ~showcase_surface_df["taxi_zone_id"].isin([264, 265])
        & showcase_surface_df["zone"].notna()
        & showcase_surface_df["zone"]
            .astype(str)
            .str.strip()
            .str.lower()
            .ne("unknown")
    )

    showcase_surface_df[
        "showcase_surface_contract_id"
    ] = SHOWCASE_SURFACE_CONTRACT_ID

    output_columns = [
        "modeling_track",
        *COMMON_FORECAST_KEY,
        "zone",
        "borough",
        "target_date",
        "target_temporal_bucket",
        "champion_family",
        "runner_up_family",
        "actual",
        "champion_prediction",
        "benchmark_prediction",
        "runner_up_prediction",
        "champion_model_available",
        "runner_up_model_available",
        "runner_up_used_benchmark_fallback",
        "diagnostic_advanced_prediction",
        "diagnostic_row_error_index",
        "benchmark_loss",
        "severe_error",
        "failure_combination",
        "low_zero_demand",
        "unstable_conditions",
        "shared_shock",
        "signed_error",
        "absolute_error",
        "squared_error",
        "benchmark_absolute_error",
        "runner_up_absolute_error",
        "champion_advantage_abs",
        "champion_better_than_benchmark",
        "job_champion_mae",
        "system_row_error_index",
        "benchmark_advantage_index",
        "reader_facing_zone",
        "showcase_surface_contract_id",
    ]

    showcase_surface_df = showcase_surface_df[output_columns]

    atomic_write_parquet(
        showcase_surface_df,
        SHOWCASE_FORECAST_RECORD_SURFACE_PATH,
    )

    del runner_df
    del diagnostic_df
    del showcase_surface_df

    showcase_surface_built_this_run = True

else:
    showcase_surface_built_this_run = False


validation_columns = [
    *PHYSICAL_FORECAST_KEY,
    "champion_prediction",
    "benchmark_prediction",
    "actual",
    "target_date",
    "reader_facing_zone",
]

showcase_validation_source_df = pd.read_parquet(
    SHOWCASE_FORECAST_RECORD_SURFACE_PATH,
    columns=validation_columns,
)

showcase_validation_df = pd.DataFrame([
    {
        "rows": len(showcase_validation_source_df),
        "jobs": len(
            showcase_validation_source_df[
                ["metric", "horizon"]
            ].drop_duplicates()
        ),
        "missing_champion_predictions":
            int(
                showcase_validation_source_df[
                    "champion_prediction"
                ].isna().sum()
            ),
        "missing_benchmark_predictions":
            int(
                showcase_validation_source_df[
                    "benchmark_prediction"
                ].isna().sum()
            ),
        "missing_actuals":
            int(showcase_validation_source_df["actual"].isna().sum()),
        "duplicate_physical_keys":
            int(
                showcase_validation_source_df.duplicated(
                    PHYSICAL_FORECAST_KEY
                ).sum()
            ),
        "min_target_date":
            showcase_validation_source_df["target_date"].min(),
        "max_target_date":
            showcase_validation_source_df["target_date"].max(),
        "reader_facing_rows":
            int(
                showcase_validation_source_df[
                    "reader_facing_zone"
                ].fillna(False).astype(bool).sum()
            ),
    }
])

del showcase_validation_source_df

surface_validation = showcase_validation_df.iloc[0]

validation_passed = (
    int(surface_validation["rows"]) == EXPECTED_SHOWCASE_ROWS
    and int(surface_validation["jobs"]) == EXPECTED_JOB_COUNT
    and int(surface_validation["missing_champion_predictions"]) == 0
    and int(surface_validation["missing_benchmark_predictions"]) == 0
    and int(surface_validation["missing_actuals"]) == 0
    and int(surface_validation["duplicate_physical_keys"]) == 0
)

if not validation_passed:
    raise RuntimeError(
        "The packaged Showcase record surface failed semantic validation."
    )

if (
    showcase_surface_built_this_run
    or not SHOWCASE_FORECAST_RECORD_AUDIT_PATH.exists()
):
    showcase_surface_audit_df = write_large_artifact_audit(
        SHOWCASE_FORECAST_RECORD_SURFACE_PATH,
        SHOWCASE_FORECAST_RECORD_AUDIT_PATH,
        SHOWCASE_SURFACE_CONTRACT_ID,
    )
else:
    showcase_surface_audit_df = pd.read_parquet(
        SHOWCASE_FORECAST_RECORD_AUDIT_PATH
    )


display(showcase_validation_df)

print(
    "PASS — Showcase forecast record surface is frozen.\n"
    f"  Rows:  {int(surface_validation['rows']):,}\n"
    f"  Jobs:  {int(surface_validation['jobs'])}\n"
    f"  Built this run: {showcase_surface_built_this_run}"
)

,rows,jobs,missing_champion_predictions,missing_benchmark_predictions,missing_actuals,duplicate_physical_keys,min_target_date,max_target_date,reader_facing_rows
0,1421436,15,0,0,0,0,2026-01-05,2026-03-31,1412724


PASS — Showcase forecast record surface is frozen.
  Rows:  1,421,436
  Jobs:  15
  Built this run: False


**Findings.** The new Showcase forecast surface reproduces the full **1,421,436-row benchmark-comparable final holdout** across all **15 Target × Horizon jobs**.

The merge introduces **0 missing champion predictions, 0 missing benchmark predictions, 0 missing observed outcomes, and 0 duplicate physical forecast keys**. The packaged evaluation period runs exactly from **January 5 through March 31, 2026**.

Of the full evaluation population, **1,412,724 rows (99.39%)** are tied to a valid physical Taxi Zone and can therefore support zone-level maps and other geographic analyses. The remaining **8,712 rows (0.61%)** correspond to Unknown or nonphysical geography. They remain in the complete analytical dataset and continue to contribute to non-geographic forecast evaluation, but they are excluded from analyses that require a valid map location.

The forecast of record remains the frozen **4.6.1 champion-system prediction**. The 4.6.2 diagnostics are joined only to add context about where and when those forecasts performed well or poorly; they do not alter the forecast values themselves.


**4.7.1.3C — Package a longer forecast history for the Showcase.**

The final holdout remains the cleanest test of how the frozen forecasting system performed on data that was not used to choose the winning models. For the interactive Showcase, though, three months is a short window for exploring recurring patterns.

This companion table follows the **same 15 frozen Chapter 4 champion jobs** back to **April 5, 2025**. It combines ordinary rolling-validation forecasts through January 4, 2026 with the existing final-holdout forecasts from January 5 through March 31, 2026.

The two periods remain labeled in the data. The earlier history gives readers a longer forecast timeline to explore; the final holdout remains the basis for the strongest out-of-sample performance claims.

The table is deliberately narrow. It preserves forecast identity, geography, time, the observed value, the champion-system forecast, the weekly benchmark, champion family, model availability, and the evaluation-period label. Detailed failure and reliability diagnostics remain attached only to the final-holdout surface.

In [17]:
# ---------------------------------------------------------------------
# 4.7.1.3C — Define the longitudinal Showcase history contract
# ---------------------------------------------------------------------

# The longitudinal table has one narrow job: let the Showcase follow the
# frozen Chapter 4 champion system farther back in time. Detailed failure
# diagnostics stay in the final-holdout surface where they were measured.
HISTORY_OUTPUT_COLUMNS = [
    "modeling_track",
    *COMMON_FORECAST_KEY,
    "zone",
    "borough",
    "target_date",
    "target_temporal_bucket",
    "champion_family",
    "actual",
    "champion_prediction",
    "benchmark_prediction",
    "champion_model_available",
    "benchmark_prediction_available",
    "evaluation_period",
    "reader_facing_zone",
    "showcase_history_contract_id",
]

HISTORY_AUDIT_REQUIRED_COLUMNS = {
    "contract_id",
    "path",
    "rows",
    "columns",
    "size_bytes",
    "modified_ns",
    "schema_signature",
    "status",
    "ordinary_validation_rows",
    "ordinary_validation_jobs",
    "ordinary_validation_taxi_zones",
    "ordinary_validation_reader_facing_rows",
    "ordinary_validation_first_target_date",
    "ordinary_validation_last_target_date",
    "final_holdout_rows",
    "final_holdout_jobs",
    "final_holdout_taxi_zones",
    "final_holdout_reader_facing_rows",
    "final_holdout_first_target_date",
    "final_holdout_last_target_date",
}

# These are the exact row-level fields shared by the three champion-family
# handoffs. Neural's observed target is handled through its frozen source
# contract rather than through runtime schema inspection.
HISTORY_VALIDATION_BASE_COLUMNS = [
    "modeling_track",
    *COMMON_FORECAST_KEY,
    "target_date",
    "prediction_raw",
    "prediction_final",
    "benchmark_prediction",
]


# ---------------------------------------------------------------------
# Freeze the same 15 ordinary Chapter 4 champions already used by 4.7.1
# ---------------------------------------------------------------------
# This is the ordinary forecast system, not the later Chapter 5 Pre-CP
# Reference roster. The expected family mix is therefore fixed at 8/4/3.
history_champion_jobs_df = (
    champion_registry_df[
        ["metric", "horizon", "champion_family", "modeling_track"]
    ]
    .drop_duplicates()
    .copy()
)

history_champion_jobs_df["horizon"] = (
    history_champion_jobs_df["horizon"].astype(int)
)
history_champion_jobs_df["champion_family"] = (
    history_champion_jobs_df["champion_family"]
    .astype(str)
    .str.strip()
    .str.lower()
)

if len(history_champion_jobs_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Showcase history requires exactly 15 frozen champion jobs."
    )

expected_champion_family_counts = {
    "tree": 8,
    "transformer": 4,
    "neural": 3,
}

observed_champion_family_counts = (
    history_champion_jobs_df["champion_family"]
    .value_counts()
    .to_dict()
)

if observed_champion_family_counts != expected_champion_family_counts:
    raise RuntimeError(
        "Frozen Chapter 4 champion composition changed: "
        f"{observed_champion_family_counts}"
    )

expected_history_job_roster = set(
    history_champion_jobs_df[
        ["metric", "horizon", "champion_family", "modeling_track"]
    ].itertuples(index=False, name=None)
)


# ---------------------------------------------------------------------
# Load the small frozen geography and target-calendar lookups
# ---------------------------------------------------------------------
# 4.1.1 stores the stable Taxi Zone identity once per Forecast Sequence.
# Reusing that compact registry gives every model family the same zone and
# borough labels without reopening the much larger target-sequence panel.
history_zone_lookup_df = pd.read_parquet(
    FORECAST_SEQUENCE_SUPPORT_REGISTRY_PATH,
    columns=[
        "forecast_sequence_id",
        "taxi_zone_id",
        "metric",
        "zone",
        "borough",
    ],
).drop_duplicates()

history_zone_lookup_df["forecast_sequence_id"] = (
    history_zone_lookup_df["forecast_sequence_id"].astype(str)
)
history_zone_lookup_df["taxi_zone_id"] = pd.to_numeric(
    history_zone_lookup_df["taxi_zone_id"],
    errors="raise",
).astype("int64")
history_zone_lookup_df["metric"] = (
    history_zone_lookup_df["metric"].astype(str)
)

history_zone_key = [
    "forecast_sequence_id",
    "taxi_zone_id",
    "metric",
]

if history_zone_lookup_df.duplicated(history_zone_key).any():
    raise RuntimeError(
        "4.1.1 support registry assigns more than one geography "
        "to a Forecast Sequence."
    )


# 4.2.1 already freezes the future target's calendar context for every
# Track × Origin × Horizon. Reusing target_temporal_bucket_y avoids rebuilding
# Daypart logic in this packaging notebook.
history_time_context_df = pd.read_parquet(
    FORECAST_TARGET_CONTEXT_PATH,
    columns=[
        "modeling_track",
        "stage",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
        "target_date",
        "target_temporal_bucket_y",
        "horizon",
    ],
).rename(
    columns={
        "target_temporal_bucket_y": "target_temporal_bucket",
    }
)

history_time_context_df = history_time_context_df.loc[
    history_time_context_df["stage"].eq("rolling_validation")
].copy()

history_time_context_df["horizon"] = (
    history_time_context_df["horizon"].astype(int)
)
history_time_context_df["origin_observation_sequence_id"] = pd.to_numeric(
    history_time_context_df["origin_observation_sequence_id"],
    errors="raise",
).astype("int64")
history_time_context_df["target_observation_sequence_id"] = pd.to_numeric(
    history_time_context_df["target_observation_sequence_id"],
    errors="raise",
).astype("int64")
history_time_context_df["target_date"] = pd.to_datetime(
    history_time_context_df["target_date"],
    errors="raise",
).dt.normalize()

history_time_context_df = history_time_context_df.loc[
    history_time_context_df["target_date"].between(
        SHOWCASE_HISTORY_VALIDATION_START_DATE,
        SHOWCASE_HISTORY_VALIDATION_END_DATE,
        inclusive="both",
    )
].copy()

history_time_key = [
    "modeling_track",
    "horizon",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
    "target_date",
]

if history_time_context_df.duplicated(history_time_key).any():
    raise RuntimeError(
        "4.2.1 target calendar contains duplicate rolling-validation "
        "target-time rows."
    )


# ---------------------------------------------------------------------
# Load one frozen champion job at a time
# ---------------------------------------------------------------------
# Reading by Target × Horizon keeps Pandas memory bounded. The filters come
# directly from the frozen champion registry, so the longer history cannot
# accidentally pull a runner-up family or a different modeling track.
def load_champion_validation_job(job):
    """
    Return one benchmark-supported ordinary-validation champion history.

    The advanced forecast is kept when it exists. If that model did not
    produce a usable forecast for a physical row, the same frozen previous-
    week benchmark used by the production system becomes the row forecast.
    """
    family = str(job.champion_family).strip().lower()
    source_contract = SHOWCASE_HISTORY_SOURCES[family]
    actual_column = source_contract["actual_column"]

    read_columns = list(
        dict.fromkeys([
            *HISTORY_VALIDATION_BASE_COLUMNS,
            actual_column,
        ])
    )

    frame = pd.read_parquet(
        require_file(
            source_contract["path"],
            f"{FAMILY_LABELS[family]} ordinary-validation predictions",
        ),
        columns=read_columns,
        filters=[
            ("modeling_track", "==", str(job.modeling_track)),
            ("metric", "==", str(job.metric)),
            ("horizon", "==", int(job.horizon)),
        ],
    )

    if frame.empty:
        raise RuntimeError(
            f"No validation rows found for {FAMILY_LABELS[family]} · "
            f"{job.metric} · h={int(job.horizon)}."
        )

    # Normalize key dtypes before joining the shared geography and target
    # calendar. This changes representation only; it does not change meaning.
    frame["forecast_sequence_id"] = (
        frame["forecast_sequence_id"].astype(str)
    )
    frame["taxi_zone_id"] = pd.to_numeric(
        frame["taxi_zone_id"],
        errors="raise",
    ).astype("int64")
    frame["origin_observation_sequence_id"] = pd.to_numeric(
        frame["origin_observation_sequence_id"],
        errors="raise",
    ).astype("int64")
    frame["target_observation_sequence_id"] = pd.to_numeric(
        frame["target_observation_sequence_id"],
        errors="raise",
    ).astype("int64")
    frame["horizon"] = frame["horizon"].astype(int)
    frame["metric"] = frame["metric"].astype(str)
    frame["target_date"] = pd.to_datetime(
        frame["target_date"],
        errors="raise",
    ).dt.normalize()

    frame = frame.loc[
        frame["target_date"].between(
            SHOWCASE_HISTORY_VALIDATION_START_DATE,
            SHOWCASE_HISTORY_VALIDATION_END_DATE,
            inclusive="both",
        )
    ].copy()

    if frame.empty:
        raise RuntimeError(
            f"No rows remain in the Showcase-history window for "
            f"{FAMILY_LABELS[family]} · {job.metric} · "
            f"h={int(job.horizon)}."
        )

    # Attach the already-frozen future target bucket. A missing match would
    # mean the prediction row and the forecasting calendar disagree.
    frame = frame.merge(
        history_time_context_df[
            [*history_time_key, "target_temporal_bucket"]
        ],
        on=history_time_key,
        how="left",
        validate="many_to_one",
    )

    if frame["target_temporal_bucket"].isna().any():
        missing_rows = int(
            frame["target_temporal_bucket"].isna().sum()
        )
        raise RuntimeError(
            f"{FAMILY_LABELS[family]} · {job.metric} · "
            f"h={int(job.horizon)} has {missing_rows:,} rows "
            "without 4.2.1 target-time context."
        )

    # Attach stable Taxi Zone labels from the canonical Forecast Sequence
    # registry so all three families use one geography definition.
    frame = frame.merge(
        history_zone_lookup_df,
        on=history_zone_key,
        how="left",
        validate="many_to_one",
    )

    if frame["zone"].isna().any():
        missing_rows = int(frame["zone"].isna().sum())
        raise RuntimeError(
            f"{FAMILY_LABELS[family]} · {job.metric} · "
            f"h={int(job.horizon)} has {missing_rows:,} rows "
            "without 4.1.1 geography context."
        )

    if actual_column != "actual":
        frame = frame.rename(
            columns={actual_column: "actual"}
        )

    for column in [
        "actual",
        "prediction_raw",
        "prediction_final",
        "benchmark_prediction",
    ]:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="coerce",
        )

    # Chapter 4 already froze the cross-family availability meaning.
    # A model is usable when both its raw and delivered predictions are
    # finite; the weekly benchmark is usable when its delivered value is finite.
    finite_actual = np.isfinite(
        frame["actual"].to_numpy(dtype=float)
    )
    model_available = (
        np.isfinite(
            frame["prediction_raw"].to_numpy(dtype=float)
        )
        & np.isfinite(
            frame["prediction_final"].to_numpy(dtype=float)
        )
    )
    benchmark_available = np.isfinite(
        frame["benchmark_prediction"].to_numpy(dtype=float)
    )

    # Keep the same benchmark-supported denominator used by the final-holdout
    # Showcase surface. Missing advanced forecasts stay in the history through
    # the production benchmark fallback rather than disappearing.
    scoreable = finite_actual & benchmark_available

    frame = frame.loc[scoreable].copy()
    model_available = model_available[scoreable]
    benchmark_available = benchmark_available[scoreable]

    frame["champion_model_available"] = model_available
    frame["benchmark_prediction_available"] = (
        benchmark_available
    )
    frame["champion_prediction"] = np.where(
        frame["champion_model_available"],
        frame["prediction_final"],
        frame["benchmark_prediction"],
    )

    if not np.isfinite(
        frame["champion_prediction"].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            f"{FAMILY_LABELS[family]} · {job.metric} · "
            f"h={int(job.horizon)} contains a non-finite system "
            "forecast after benchmark fallback."
        )

    # Preserve the same map-facing rule used by the existing holdout surface.
    # Unknown/nonphysical zones stay in the analytical table but are flagged
    # out of views that require a real Taxi Zone location.
    zone_text = (
        frame["zone"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    frame["reader_facing_zone"] = (
        ~frame["taxi_zone_id"].isin([264, 265])
        & frame["zone"].notna()
        & zone_text.ne("unknown")
    )

    frame["champion_family"] = family
    frame["evaluation_period"] = "ordinary_validation"
    frame["showcase_history_contract_id"] = (
        SHOWCASE_HISTORY_CONTRACT_ID
    )

    result = frame[HISTORY_OUTPUT_COLUMNS].copy()

    if result.duplicated(PHYSICAL_FORECAST_KEY).any():
        raise RuntimeError(
            f"{FAMILY_LABELS[family]} · {job.metric} · "
            f"h={int(job.horizon)} contains duplicate physical "
            "forecast rows."
        )

    return result


# ---------------------------------------------------------------------
# Cheap recovery for a previously validated longitudinal surface
# ---------------------------------------------------------------------
# The large row-level file is fully validated when created. Later notebook
# reruns use its small audit marker rather than scanning millions of rows just
# to decide whether the artifact can be reused.
def showcase_history_artifact_is_reusable():
    """
    Return True only for a complete current-contract history artifact.

    File metadata protects against a changed or partial Parquet file, while the
    tiny audit also preserves the period-level facts needed by final QA.
    """
    if (
        not SHOWCASE_FORECAST_HISTORY_SURFACE_PATH.exists()
        or not SHOWCASE_FORECAST_HISTORY_AUDIT_PATH.exists()
    ):
        return False

    audit_df = pd.read_parquet(
        SHOWCASE_FORECAST_HISTORY_AUDIT_PATH
    )

    if len(audit_df) != 1:
        return False

    if not HISTORY_AUDIT_REQUIRED_COLUMNS.issubset(
        audit_df.columns
    ):
        return False

    return large_artifact_matches_audit(
        SHOWCASE_FORECAST_HISTORY_SURFACE_PATH,
        SHOWCASE_FORECAST_HISTORY_AUDIT_PATH,
        SHOWCASE_HISTORY_CONTRACT_ID,
    )

In [18]:
# ---------------------------------------------------------------------
# 4.7.1.3C — Build or recover the longitudinal Showcase history
# ---------------------------------------------------------------------

history_surface_reusable = (
    showcase_history_artifact_is_reusable()
)

if (
    RUN_SHOWCASE_HISTORY_BUILD
    and (
        REBUILD_SHOWCASE_HISTORY
        or not history_surface_reusable
    )
):
    # Build only the 15 frozen champion jobs. One-job-at-a-time reads keep
    # memory modest and make any broken upstream alignment easy to identify.
    validation_history_frames = []

    for job in history_champion_jobs_df.itertuples(
        index=False
    ):
        print(
            "Loading validation history · "
            f"{FAMILY_LABELS[job.champion_family]} · "
            f"{job.metric} · h={int(job.horizon)}",
            flush=True,
        )

        validation_history_frames.append(
            load_champion_validation_job(job)
        )

    validation_history_df = pd.concat(
        validation_history_frames,
        ignore_index=True,
    )

    del validation_history_frames


    # -----------------------------------------------------------------
    # Carry forward the already-frozen final holdout
    # -----------------------------------------------------------------
    # The holdout portion already exists as a validated 4.7.1 artifact.
    # Reusing it directly guarantees that the longitudinal package cannot
    # invent a second version of the January–March 2026 forecast record.
    holdout_columns = [
        "modeling_track",
        *COMMON_FORECAST_KEY,
        "zone",
        "borough",
        "target_date",
        "target_temporal_bucket",
        "champion_family",
        "actual",
        "champion_prediction",
        "benchmark_prediction",
        "champion_model_available",
        "reader_facing_zone",
    ]

    holdout_history_df = pd.read_parquet(
        SHOWCASE_FORECAST_RECORD_SURFACE_PATH,
        columns=holdout_columns,
    )

    holdout_history_df["horizon"] = (
        holdout_history_df["horizon"].astype(int)
    )
    holdout_history_df["target_date"] = pd.to_datetime(
        holdout_history_df["target_date"],
        errors="raise",
    ).dt.normalize()

    holdout_history_df = holdout_history_df.loc[
        holdout_history_df["target_date"].between(
            SHOWCASE_HISTORY_HOLDOUT_START_DATE,
            SHOWCASE_HISTORY_HOLDOUT_END_DATE,
            inclusive="both",
        )
    ].copy()

    if len(holdout_history_df) != EXPECTED_SHOWCASE_ROWS:
        raise RuntimeError(
            "The longitudinal history no longer reproduces the "
            "complete 1,421,436-row final-holdout Showcase surface."
        )

    holdout_history_df["champion_family"] = (
        holdout_history_df["champion_family"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    holdout_history_df[
        "benchmark_prediction_available"
    ] = np.isfinite(
        pd.to_numeric(
            holdout_history_df["benchmark_prediction"],
            errors="coerce",
        ).to_numpy(dtype=float)
    )

    holdout_history_df["evaluation_period"] = (
        "final_holdout"
    )
    holdout_history_df[
        "showcase_history_contract_id"
    ] = SHOWCASE_HISTORY_CONTRACT_ID

    holdout_history_df = holdout_history_df[
        HISTORY_OUTPUT_COLUMNS
    ].copy()


    # -----------------------------------------------------------------
    # Normalize the two evidence periods to one durable output contract
    # -----------------------------------------------------------------
    # Validation and holdout come from different Parquet producers. Their
    # analytical fields mean the same thing, but their in-memory Pandas
    # representations do not need to be identical. Normalize the narrow
    # output contract once, then write each period separately rather than
    # creating another nearly six-million-row DataFrame in memory.

    history_string_columns = [
        "modeling_track",
        "metric",
        "forecast_sequence_id",
        "zone",
        "borough",
        "target_temporal_bucket",
        "champion_family",
        "evaluation_period",
        "showcase_history_contract_id",
    ]

    history_integer_columns = [
        "horizon",
        "taxi_zone_id",
        "origin_observation_sequence_id",
        "target_observation_sequence_id",
    ]

    history_float_columns = [
        "actual",
        "champion_prediction",
        "benchmark_prediction",
    ]

    history_boolean_columns = [
        "champion_model_available",
        "benchmark_prediction_available",
        "reader_facing_zone",
    ]


    def normalize_history_dtypes(frame):
        """
        Give both evidence periods one explicit output representation.

        This changes storage representation only. Forecast values, geography,
        timing, model identity, and evaluation-period meaning remain unchanged.
        """
        result = frame.loc[:, HISTORY_OUTPUT_COLUMNS].copy()

        for column in history_string_columns:
            result[column] = result[column].astype("string")

        for column in history_integer_columns:
            result[column] = pd.to_numeric(
                result[column],
                errors="raise",
            ).astype("int64")

        for column in history_float_columns:
            result[column] = pd.to_numeric(
                result[column],
                errors="raise",
            ).astype("float64")

        for column in history_boolean_columns:
            result[column] = result[column].astype(bool)

        result["target_date"] = pd.to_datetime(
            result["target_date"],
            errors="raise",
        ).dt.normalize()

        return result


    validation_history_df = normalize_history_dtypes(
        validation_history_df
    )
    holdout_history_df = normalize_history_dtypes(
        holdout_history_df
    )


    # -----------------------------------------------------------------
    # Validate each period before anything is written
    # -----------------------------------------------------------------
    # The periods have different row counts by design. What must agree is the
    # frozen 15-job roster, the intended date coverage, row uniqueness, and
    # the availability of observed/forecast/benchmark values.

    expected_period_bounds = {
        "ordinary_validation": (
            SHOWCASE_HISTORY_VALIDATION_START_DATE,
            SHOWCASE_HISTORY_VALIDATION_END_DATE,
        ),
        "final_holdout": (
            SHOWCASE_HISTORY_HOLDOUT_START_DATE,
            SHOWCASE_HISTORY_HOLDOUT_END_DATE,
        ),
    }


    def validate_history_period(
        frame,
        *,
        period,
        expected_start,
        expected_end,
    ):
        """
        Validate one evidence period and return its compact audit summary.

        Keeping validation period-specific avoids needing a giant combined
        DataFrame merely to verify roster, dates, values, and uniqueness.
        """
        if frame.empty:
            raise RuntimeError(
                f"{period} contains no forecast-history rows."
            )

        if not frame["evaluation_period"].eq(period).all():
            raise RuntimeError(
                f"{period} contains rows labeled as another evidence period."
            )

        if frame.duplicated(PHYSICAL_FORECAST_KEY).any():
            raise RuntimeError(
                f"{period} contains duplicate physical forecast rows."
            )

        observed_roster = set(
            frame[
                [
                    "metric",
                    "horizon",
                    "champion_family",
                    "modeling_track",
                ]
            ]
            .drop_duplicates()
            .itertuples(index=False, name=None)
        )

        if observed_roster != expected_history_job_roster:
            raise RuntimeError(
                f"{period} does not reproduce the frozen "
                "15-job Chapter 4 champion roster."
            )

        # Each Target × Horizon must span the full requested period. A family
        # with shorter coverage cannot hide behind another complete family.
        job_bounds_df = (
            frame
            .groupby(
                [
                    "metric",
                    "horizon",
                    "champion_family",
                    "modeling_track",
                ],
                observed=True,
                as_index=False,
            )
            .agg(
                first_target_date=("target_date", "min"),
                last_target_date=("target_date", "max"),
            )
        )

        # Every frozen job should enter the evaluation period at the expected start.
        # The final day can differ slightly by target because the frozen forecasting
        # surface only contains rows that were actually supported for that target.
        # We therefore require the PERIOD to reach the expected end date rather than
        # inventing missing rows for a job whose valid support ends one day earlier.

        job_starts_ok = (
            job_bounds_df["first_target_date"]
            .eq(expected_start)
            .all()
        )

        job_ends_within_period = (
            job_bounds_df["last_target_date"]
            .le(expected_end)
            .all()
        )

        period_reaches_expected_end = (
            frame["target_date"].max()
            == expected_end
        )

        if not (
            job_starts_ok
            and job_ends_within_period
            and period_reaches_expected_end
        ):
            bad_bounds = job_bounds_df.loc[
                ~job_bounds_df["first_target_date"].eq(
                    expected_start
                )
                | job_bounds_df["last_target_date"].gt(
                    expected_end
                )
            ]

            raise RuntimeError(
                f"{period} does not preserve the intended evaluation window.\n"
                f"Expected start: {expected_start.date()}\n"
                f"Expected period end: {expected_end.date()}\n"
                f"Observed period end: {frame['target_date'].max().date()}\n"
                f"Jobs outside the allowed bounds:\n"
                f"{bad_bounds.to_string(index=False)}"
            )
            
        # Genuine observed zero activity is valid. Only missing or non-finite
        # numerical values are rejected.
        core_values_are_finite = np.isfinite(
            frame[
                [
                    "actual",
                    "champion_prediction",
                    "benchmark_prediction",
                ]
            ].to_numpy(dtype=float)
        ).all()

        if not core_values_are_finite:
            raise RuntimeError(
                f"{period} contains a non-finite observed, "
                "champion, or benchmark value."
            )

        return {
            "evaluation_period": period,
            "rows": len(frame),
            "jobs": len(observed_roster),
            "taxi_zones": frame["taxi_zone_id"].nunique(),
            "reader_facing_rows": int(
                frame["reader_facing_zone"].sum()
            ),
            "first_target_date": frame["target_date"].min(),
            "last_target_date": frame["target_date"].max(),
        }


    validation_summary = validate_history_period(
        validation_history_df,
        period="ordinary_validation",
        expected_start=SHOWCASE_HISTORY_VALIDATION_START_DATE,
        expected_end=SHOWCASE_HISTORY_VALIDATION_END_DATE,
    )

    holdout_summary = validate_history_period(
        holdout_history_df,
        period="final_holdout",
        expected_start=SHOWCASE_HISTORY_HOLDOUT_START_DATE,
        expected_end=SHOWCASE_HISTORY_HOLDOUT_END_DATE,
    )

    if holdout_summary["rows"] != EXPECTED_SHOWCASE_ROWS:
        raise RuntimeError(
            "Final-holdout history no longer reproduces the complete "
            f"{EXPECTED_SHOWCASE_ROWS:,}-row Showcase evaluation surface."
        )


    # -----------------------------------------------------------------
    # Confirm that the two periods cannot collide on physical forecast keys
    # -----------------------------------------------------------------
    # PHYSICAL_FORECAST_KEY contains the target observation sequence. Those
    # sequence IDs should progress forward with time, so the validation target
    # range must end before the holdout target range begins.
    validation_last_sequence = int(
        validation_history_df[
            "target_observation_sequence_id"
        ].max()
    )
    holdout_first_sequence = int(
        holdout_history_df[
            "target_observation_sequence_id"
        ].min()
    )

    if validation_last_sequence >= holdout_first_sequence:
        raise RuntimeError(
            "Validation and holdout target-sequence ranges overlap: "
            f"{validation_last_sequence} vs {holdout_first_sequence}."
        )


    # -----------------------------------------------------------------
    # Write both periods sequentially into one Parquet artifact
    # -----------------------------------------------------------------
    # Parquet supports multiple row groups in one file. Writing validation
    # first and holdout second produces one longitudinal table without asking
    # Pandas to allocate a second six-million-row combined DataFrame.
    temporary_history_path = (
        SHOWCASE_FORECAST_HISTORY_SURFACE_PATH
        .with_suffix(".tmp.parquet")
    )

    if temporary_history_path.exists():
        temporary_history_path.unlink()

    validation_table = pa.Table.from_pandas(
        validation_history_df,
        preserve_index=False,
    )

    # Remove Pandas-specific metadata so the same explicit Arrow schema can be
    # used for both source periods.
    history_arrow_schema = (
        validation_table.schema.remove_metadata()
    )
    validation_table = validation_table.cast(
        history_arrow_schema
    )

    holdout_table = pa.Table.from_pandas(
        holdout_history_df,
        preserve_index=False,
    ).cast(
        history_arrow_schema
    )

    writer = pq.ParquetWriter(
        temporary_history_path,
        history_arrow_schema,
        compression="zstd",
    )

    try:
        writer.write_table(
            validation_table,
            row_group_size=100_000,
        )
        writer.write_table(
            holdout_table,
            row_group_size=100_000,
        )
    finally:
        writer.close()

    expected_history_rows = (
        validation_summary["rows"]
        + holdout_summary["rows"]
    )

    # Check the completed temporary file before it is allowed to replace the
    # durable current-contract artifact.
    written_rows = int(
        pq.ParquetFile(
            temporary_history_path
        ).metadata.num_rows
    )

    if written_rows != expected_history_rows:
        temporary_history_path.unlink(
            missing_ok=True
        )
        raise RuntimeError(
            "Longitudinal history write changed the row count: "
            f"{written_rows:,} vs {expected_history_rows:,}."
        )

    temporary_history_path.replace(
        SHOWCASE_FORECAST_HISTORY_SURFACE_PATH
    )

    del validation_table
    del holdout_table


    # -----------------------------------------------------------------
    # Freeze a compact period-level audit beside the large row-level file
    # -----------------------------------------------------------------
    # The two small summaries contain everything later QA needs, so ordinary
    # notebook reruns do not have to scan the six-million-row history again.
    history_period_summary_df = pd.DataFrame(
        [
            validation_summary,
            holdout_summary,
        ]
    )

    history_surface_audit_df = (
        write_large_artifact_audit(
            SHOWCASE_FORECAST_HISTORY_SURFACE_PATH,
            SHOWCASE_FORECAST_HISTORY_AUDIT_PATH,
            SHOWCASE_HISTORY_CONTRACT_ID,
        )
    )

    for period in [
        "ordinary_validation",
        "final_holdout",
    ]:
        period_summary = (
            history_period_summary_df.loc[
                history_period_summary_df[
                    "evaluation_period"
                ].eq(period)
            ].iloc[0]
        )

        history_surface_audit_df.loc[
            0,
            f"{period}_rows",
        ] = int(period_summary["rows"])

        history_surface_audit_df.loc[
            0,
            f"{period}_jobs",
        ] = int(period_summary["jobs"])

        history_surface_audit_df.loc[
            0,
            f"{period}_taxi_zones",
        ] = int(period_summary["taxi_zones"])

        history_surface_audit_df.loc[
            0,
            f"{period}_reader_facing_rows",
        ] = int(
            period_summary["reader_facing_rows"]
        )

        history_surface_audit_df.loc[
            0,
            f"{period}_first_target_date",
        ] = period_summary["first_target_date"]

        history_surface_audit_df.loc[
            0,
            f"{period}_last_target_date",
        ] = period_summary["last_target_date"]

    atomic_write_parquet(
        history_surface_audit_df,
        SHOWCASE_FORECAST_HISTORY_AUDIT_PATH,
    )

    print(
        "PASS — longer Showcase forecast history packaged.\n"
        f"  Rows: {expected_history_rows:,}\n"
        f"  Ordinary-validation rows: "
        f"{validation_summary['rows']:,}\n"
        f"  Final-holdout rows: "
        f"{holdout_summary['rows']:,}\n"
        f"  Jobs: {EXPECTED_JOB_COUNT}\n"
        f"  Output: {SHOWCASE_FORECAST_HISTORY_SURFACE_PATH}",
        flush=True,
    )

    del validation_history_df
    del holdout_history_df

elif history_surface_reusable:
    print(
        "PASS — reusing current longer Showcase forecast history.",
        flush=True,
    )

else:
    raise RuntimeError(
        "The longer Showcase forecast history is missing or "
        "stale, but RUN_SHOWCASE_HISTORY_BUILD is False."
    )


# ---------------------------------------------------------------------
# Show only the compact period summary needed for notebook review
# ---------------------------------------------------------------------
# The row-level table can contain millions of records. Keeping the notebook
# output to two summary rows makes the result reviewable without dumping data.
history_surface_audit_df = pd.read_parquet(
    SHOWCASE_FORECAST_HISTORY_AUDIT_PATH
)

require_columns(
    history_surface_audit_df,
    HISTORY_AUDIT_REQUIRED_COLUMNS,
    "Showcase forecast-history audit",
)

history_audit = history_surface_audit_df.iloc[0]

history_period_summary_df = pd.DataFrame([
    {
        "Period": "Ordinary validation",
        "Rows": int(
            history_audit[
                "ordinary_validation_rows"
            ]
        ),
        "Jobs": int(
            history_audit[
                "ordinary_validation_jobs"
            ]
        ),
        "Taxi Zones": int(
            history_audit[
                "ordinary_validation_taxi_zones"
            ]
        ),
        "Reader-facing rows": int(
            history_audit[
                "ordinary_validation_reader_facing_rows"
            ]
        ),
        "First target date": pd.Timestamp(
            history_audit[
                "ordinary_validation_first_target_date"
            ]
        ).date(),
        "Last target date": pd.Timestamp(
            history_audit[
                "ordinary_validation_last_target_date"
            ]
        ).date(),
    },
    {
        "Period": "Final holdout",
        "Rows": int(
            history_audit[
                "final_holdout_rows"
            ]
        ),
        "Jobs": int(
            history_audit[
                "final_holdout_jobs"
            ]
        ),
        "Taxi Zones": int(
            history_audit[
                "final_holdout_taxi_zones"
            ]
        ),
        "Reader-facing rows": int(
            history_audit[
                "final_holdout_reader_facing_rows"
            ]
        ),
        "First target date": pd.Timestamp(
            history_audit[
                "final_holdout_first_target_date"
            ]
        ).date(),
        "Last target date": pd.Timestamp(
            history_audit[
                "final_holdout_last_target_date"
            ]
        ).date(),
    },
])

display(history_period_summary_df)

PASS — reusing current longer Showcase forecast history.


,Period,Rows,Jobs,Taxi Zones,Reader-facing rows,First target date,Last target date
0,Ordinary validation,4558683,15,258,4530489,2025-04-05,2026-01-04
1,Final holdout,1421436,15,258,1412724,2026-01-05,2026-03-31


**Findings.** The Showcase forecast history contains **5,980,119 rows** across all **15 Target × Horizon jobs** and **258 Taxi Zones**.

The ordinary-validation period spans **April 5, 2025 through January 4, 2026** with **4,558,683 rows**. The final-holdout period spans **January 5 through March 31, 2026** with **1,421,436 rows**.

The two periods remain explicitly separated so the longer timeline can support forecast exploration while the final holdout remains the primary basis for evaluating out-of-sample forecast performance.

## 4.7.1.4 Build Compact Final-Holdout Showcase Summaries

The detailed final-holdout row surface is appropriate for interactive filtering and Forecast Wins & Misses, but most scorecard and reliability views should not scan 1.4 million rows merely to draw a headline chart.

This section therefore pre-aggregates the **final holdout only** at several common grains:

* Target × Horizon;
* Taxi Zone × Target × Horizon;
* Temporal Bucket × Target × Horizon; and
* Date × Target × Horizon.

The longer history added above is kept separate for timeline exploration. These compact summaries continue to use the frozen unseen holdout so their accuracy and reliability meaning does not change.

**4.7.1.4A — Build compact accuracy and reliability summaries.**

Native-unit MAE remains available for interpretation within a target. For comparisons across Taxi trips, vehicle speeds, FHVHV demand, and Subway ridership, the summaries also preserve **Relative MAE**:

$$
\text{Relative MAE (\%)} =
100 \times
\frac{\sum |\hat{y} - y|}
{\sum |y|}
$$

Benchmark skill is computed on the same rows:

$$
\text{Benchmark Skill (\%)} =
100 \times
\left(
1 -
\frac{\text{Champion MAE}}
{\text{Benchmark MAE}}
\right)
$$

Positive benchmark skill favors the frozen champion system; negative skill favors the weekly benchmark.

In [19]:
# ---------------------------------------------------------------------
# 4.7.1.4A — Build compact Showcase summary surfaces
# ---------------------------------------------------------------------

summary_outputs_ready = all(
    path.exists()
    for path in [
        SHOWCASE_JOB_SUMMARY_PATH,
        SHOWCASE_ZONE_SUMMARY_PATH,
        SHOWCASE_TEMPORAL_BUCKET_SUMMARY_PATH,
        SHOWCASE_DATE_SUMMARY_PATH,
    ]
)

if REBUILD_SHOWCASE_SUMMARIES or not summary_outputs_ready:
    summary_source_columns = [
        "metric",
        "horizon",
        "champion_family",
        "taxi_zone_id",
        "zone",
        "borough",
        "reader_facing_zone",
        "target_temporal_bucket",
        "target_date",
        "actual",
        "champion_prediction",
        "benchmark_prediction",
        "champion_better_than_benchmark",
        "benchmark_loss",
        "severe_error",
        "system_row_error_index",
        "benchmark_advantage_index",
    ]

    showcase_summary_source_df = pd.read_parquet(
        SHOWCASE_FORECAST_RECORD_SURFACE_PATH,
        columns=summary_source_columns,
    )

    summary_specs = [
        (
            SHOWCASE_JOB_SUMMARY_PATH,
            [
                "metric",
                "horizon",
                "champion_family",
            ],
        ),
        (
            SHOWCASE_ZONE_SUMMARY_PATH,
            [
                "metric",
                "horizon",
                "champion_family",
                "taxi_zone_id",
                "zone",
                "borough",
                "reader_facing_zone",
            ],
        ),
        (
            SHOWCASE_TEMPORAL_BUCKET_SUMMARY_PATH,
            [
                "metric",
                "horizon",
                "champion_family",
                "target_temporal_bucket",
            ],
        ),
        (
            SHOWCASE_DATE_SUMMARY_PATH,
            [
                "metric",
                "horizon",
                "champion_family",
                "target_date",
            ],
        ),
    ]

    for output_path, group_columns in summary_specs:
        summary_df = summarize_showcase_surface(
            showcase_summary_source_df,
            group_columns,
        )

        atomic_write_parquet(
            summary_df,
            output_path,
        )

    del showcase_summary_source_df
    del summary_df


showcase_job_summary_df = pd.read_parquet(
    SHOWCASE_JOB_SUMMARY_PATH
)

if len(showcase_job_summary_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Showcase job summary must contain exactly 15 rows."
    )


display(
    showcase_job_summary_df.assign(
        Target=lambda frame: frame["metric"].map(TARGET_LABELS),
        Champion=lambda frame: frame["champion_family"].map(FAMILY_LABELS),
    )[
        [
            "Target",
            "horizon",
            "Champion",
            "forecast_rows",
            "mae",
            "relative_mae_pct",
            "benchmark_skill_pct",
            "benchmark_loss_pct",
            "severe_error_pct",
        ]
    ].sort_values(["Target", "horizon"]).round(3)
)

,Target,horizon,Champion,forecast_rows,mae,relative_mae_pct,benchmark_skill_pct,benchmark_loss_pct,severe_error_pct
3,FHVHV speed,1,Transformer,110012,0.954,5.835,36.460,33.698,10.001
13,FHVHV speed,2,Transformer,110012,1.014,6.205,32.428,35.471,10.001
14,FHVHV speed,5,Transformer,110012,1.041,6.369,30.638,36.507,10.001
10,FHVHV trips,1,Tree,110499,57.622,10.556,37.665,41.723,10.000
6,FHVHV trips,2,Tree,110499,65.738,12.043,28.885,44.869,10.000
8,FHVHV trips,5,Tree,110499,66.102,12.109,28.491,46.450,10.000
0,Subway ridership,1,Tree,64477,407.487,8.841,35.836,49.016,10.000
4,Subway ridership,2,Transformer,64477,512.990,11.130,19.223,59.446,10.000
2,Subway ridership,5,Tree,64477,454.958,9.871,28.361,48.335,10.000
12,Taxi speed,1,Neural,78325,2.137,16.013,28.525,37.035,10.001


**Findings.** The frozen champion system beats its simple benchmark **in aggregate for all 15 Target × Horizon jobs**, with benchmark-relative MAE improvements ranging from **19.2% to 37.7%**.

The strongest benchmark-relative gains appear for **FHVHV trips at h=1 (+37.7%)**, **FHVHV speed at h=1 (+36.5%)**, and **Subway ridership at h=1 (+35.8%)**. The weakest job is **Subway ridership at h=2 (+19.2%)**, consistent with the forecasting difficulty already identified upstream.

Relative MAE also makes the cross-target scale differences much clearer. FHVHV speed is the most accurate target proportionally at roughly **5.8–6.4% Relative MAE**, followed by Subway ridership at roughly **8.8–11.1%** and FHVHV trips at **10.6–12.1%**. Taxi trips are higher at approximately **14.7–15.8%**, while Taxi speed is the most difficult proportionally at approximately **16.0–16.3%**.

Longer horizons generally degrade performance, but not perfectly monotonically for every target. The packaged summaries therefore preserve the full Target × Horizon structure rather than reducing forecasting quality to one overall score.

**4.7.1.4B — Confirm that Wins & Misses can be ranked fairly across targets.**

A large native-unit error is not automatically an important miss. Missing 500 Subway riders and missing 500 Taxi trips represent very different forecasting problems.

The record surface therefore includes two job-normalized measures:

* `system_row_error_index` — how large this row's champion error is relative to its Target × Horizon's typical champion error; and
* `benchmark_advantage_index` — how much the champion beats or trails the benchmark, scaled by that same job-level error magnitude.

These measures are intended for **ranking examples across targets**. The displayed examples should still show the observed and predicted values in their native units so a reader can understand what actually happened.

In [20]:
# ---------------------------------------------------------------------
# 4.7.1.4B — Audit scale-aware ranking support
# ---------------------------------------------------------------------

ranking_source_df = pd.read_parquet(
    SHOWCASE_FORECAST_RECORD_SURFACE_PATH,
    columns=[
        "metric",
        "horizon",
        "reader_facing_zone",
        "benchmark_advantage_index",
        "system_row_error_index",
    ],
)

ranking_source_df["reader_facing_zone"] = (
    ranking_source_df["reader_facing_zone"]
    .fillna(False)
    .astype(bool)
)

ranking_source_df["_rankable_win"] = (
    ranking_source_df["reader_facing_zone"]
    & ranking_source_df["benchmark_advantage_index"].gt(0)
)

ranking_source_df["_rankable_miss"] = (
    ranking_source_df["reader_facing_zone"]
    & ranking_source_df["system_row_error_index"].notna()
)

ranking_source_df["_reader_win_index"] = (
    ranking_source_df["benchmark_advantage_index"]
    .where(ranking_source_df["reader_facing_zone"])
)

ranking_source_df["_reader_miss_index"] = (
    ranking_source_df["system_row_error_index"]
    .where(ranking_source_df["reader_facing_zone"])
)

ranking_support_df = (
    ranking_source_df.groupby(
        ["metric", "horizon"],
        observed=True,
    )
    .agg(
        forecast_rows=("metric", "size"),
        reader_facing_rows=("reader_facing_zone", "sum"),
        rankable_win_rows=("_rankable_win", "sum"),
        rankable_miss_rows=("_rankable_miss", "sum"),
        max_win_index=("_reader_win_index", "max"),
        max_miss_index=("_reader_miss_index", "max"),
    )
    .reset_index()
    .sort_values(["metric", "horizon"])
    .reset_index(drop=True)
)

del ranking_source_df


if len(ranking_support_df) != EXPECTED_JOB_COUNT:
    raise RuntimeError(
        "Scale-aware Showcase ranking audit must cover all 15 jobs."
    )

if (
    ranking_support_df["rankable_win_rows"].le(0).any()
    or ranking_support_df["rankable_miss_rows"].le(0).any()
):
    raise RuntimeError(
        "At least one Target × Horizon lacks ranking support."
    )


display(
    ranking_support_df.assign(
        Target=lambda frame: frame["metric"].map(TARGET_LABELS)
    )[
        [
            "Target",
            "horizon",
            "reader_facing_rows",
            "rankable_win_rows",
            "rankable_miss_rows",
            "max_win_index",
            "max_miss_index",
        ]
    ].round(3)
)

,Target,horizon,reader_facing_rows,rankable_win_rows,rankable_miss_rows,max_win_index,max_miss_index
0,FHVHV speed,1,109601,72703,109601,3011.835,2952.590
1,FHVHV speed,2,109601,70736,109601,3003.562,2921.925
2,FHVHV speed,5,109601,69591,109601,2858.108,2886.480
3,FHVHV trips,1,109655,63915,109655,13005.328,4733.941
4,FHVHV trips,2,109655,60615,109655,11330.886,6127.645
5,FHVHV trips,5,109655,58505,109655,10838.884,7122.925
6,Subway ridership,1,64477,32870,64477,13278.377,8873.666
7,Subway ridership,2,64477,26148,64477,9902.730,8276.766
8,Subway ridership,5,64477,33310,64477,11348.239,9843.983
9,Taxi speed,1,77520,48820,77520,2128.783,2618.174


**Findings.** All **15 Target × Horizon jobs** contain substantial numbers of valid physical-zone forecasts that can be used to identify both **Forecast Wins** and **Forecast Misses**.

Every job contains tens of thousands of forecasts that can be ranked by the scale-aware miss-severity index, as well as a large number of forecasts where the champion model outperforms the weekly benchmark. Because these rankings are normalized within each Target × Horizon, the Showcase can compare strong and weak forecast examples across Taxi trips, Subway ridership, and average speeds without allowing their very different native units to dominate the ranking.

These scale-aware indices are used only to make the examples comparable across targets. Any selected example should still show the original observed value, champion forecast, benchmark forecast, location, time, and diagnostic context so the underlying mobility result remains interpretable.


### Section Summary

The future Showcase no longer needs to reconstruct forecasting outputs from raw model files.

The detailed final-holdout surface preserves all **1,421,436** benchmark-comparable evaluation records and their diagnostic context. A separate longitudinal history follows the same frozen 15-job champion system from **April 5, 2025 through March 31, 2026**, with ordinary validation and final holdout labeled explicitly.

Compact Job, Taxi Zone, Temporal Bucket, and Date summaries remain tied to the final holdout. Forecast Wins & Misses therefore keep their clean evaluation meaning, while the interactive forecast explorer gains a longer history for investigation.

Unknown or nonphysical Taxi Zones remain in the complete analytical artifacts but are marked so they can be excluded from maps and other views that require a valid geographic location.

## 4.7.1.5 Freeze the Chapter 4 Forecasting Handoff

The final section checks the package as a whole.

A successful handoff must establish seven things:

1. all four frozen forecasting families are represented completely;
2. Chapter 5 still has **no selected Pre-CP winner**;
3. the Pre-CP time, support, benchmark, and recursive-application contracts remain intact;
4. the final-holdout Showcase surface still reproduces the frozen benchmark-comparable evaluation set;
5. the longer Showcase forecast history is present with ordinary validation and final holdout kept distinct;
6. 4.6.2 diagnostic cautions remain explanatory rather than exclusion rules; and
7. every file advertised by the final manifests exists and is readable.

If those checks pass, Chapter 4 forecasting can close without forcing later notebooks to reconstruct its internal state.

**4.7.1.5A — Publish the Showcase handoff directory.**

4.7.1 owns two complementary row-level Showcase products: the detailed final-holdout evaluation surface and the longer champion-forecast history. The four compact accuracy and reliability summaries remain final-holdout products.

Existing reliability, failure-analysis, interpretation, and robustness evidence remain at their authoritative 4.6.1/4.6.2 locations and are referenced rather than copied here.

The larger 1,427,340-row 4.6.1 champion holdout artifact is also preserved by reference. That keeps the complete audit trail available even though the app-facing evaluation surface intentionally uses the 1,421,436-row benchmark-supported intersection.

In [21]:
# ---------------------------------------------------------------------
# 4.7.1.5A — Publish the Showcase handoff manifest
# ---------------------------------------------------------------------

showcase_handoff_specs = [
    (
        "showcase_forecast_record_surface",
        SHOWCASE_FORECAST_RECORD_SURFACE_PATH,
        "Benchmark-comparable row-level forecast surface",
        "Showcase",
        "4.7.1_owned",
        "1,421,436 observed/champion/benchmark/runner-up records with diagnostics.",
    ),
    (
        "showcase_forecast_history_surface",
        SHOWCASE_FORECAST_HISTORY_SURFACE_PATH,
        "Longitudinal champion forecast history",
        "Showcase",
        "4.7.1_owned",
        (
            "April 2025 through March 2026 champion-system trajectories "
            "with ordinary-validation versus final-holdout provenance."
        ),
    ),
    (
        "showcase_forecast_job_summary",
        SHOWCASE_JOB_SUMMARY_PATH,
        "Target × Horizon summary",
        "Showcase",
        "4.7.1_owned",
        "Headline native-unit, relative-error, and benchmark-skill metrics.",
    ),
    (
        "showcase_forecast_zone_summary",
        SHOWCASE_ZONE_SUMMARY_PATH,
        "Taxi Zone reliability summary",
        "Showcase",
        "4.7.1_owned",
        "Zone-level forecast performance for maps and rankings.",
    ),
    (
        "showcase_forecast_temporal_bucket_summary",
        SHOWCASE_TEMPORAL_BUCKET_SUMMARY_PATH,
        "Temporal Bucket reliability summary",
        "Showcase",
        "4.7.1_owned",
        "Daypart-pattern forecasting performance.",
    ),
    (
        "showcase_forecast_date_summary",
        SHOWCASE_DATE_SUMMARY_PATH,
        "Daily reliability summary",
        "Showcase",
        "4.7.1_owned",
        "Calendar-date forecast performance and difficult-period views.",
    ),
    (
        "complete_champion_holdout_predictions",
        handoff_461_paths["champion_holdout_predictions"],
        "Complete 4.6.1 holdout audit surface",
        "Audit / advanced Showcase",
        "upstream_reference",
        "Larger authoritative holdout artifact including rows without benchmark support.",
    ),
    (
        "chapter4_champion_registry",
        handoff_461_paths["champion_registry"],
        "Frozen ordinary champion system",
        "Showcase",
        "upstream_reference",
        "Validation-selected champion family for each Target × Horizon.",
    ),
    (
        "chapter4_runner_up_registry",
        handoff_461_paths["pre_holdout_runner_up_registry"],
        "Frozen runner-up identities",
        "Showcase",
        "upstream_reference",
        "Pre-holdout alternatives used for robustness reporting.",
    ),
]

for row in handoff_462_df.itertuples(index=False):
    if str(row.artifact_id) == "champion_holdout_predictions":
        continue

    showcase_handoff_specs.append(
        (
            f"4.6.2__{row.artifact_id}",
            Path(str(row.path)),
            "4.6.2 evaluation evidence",
            "Showcase / audit",
            "upstream_reference",
            str(row.purpose),
        )
    )


showcase_handoff_manifest_df = pd.DataFrame([
    artifact_inventory_row(
        artifact_id,
        path,
        role,
        consumer,
        ownership,
        purpose,
    )
    for artifact_id, path, role, consumer, ownership, purpose
    in showcase_handoff_specs
])

if showcase_handoff_manifest_df["artifact_id"].duplicated().any():
    raise RuntimeError(
        "Showcase forecasting handoff contains duplicate artifact IDs."
    )

if WRITE_FINAL_OUTPUTS:
    atomic_write_parquet(
        showcase_handoff_manifest_df,
        SHOWCASE_HANDOFF_MANIFEST_PATH,
    )


display(
    showcase_handoff_manifest_df.groupby(
        ["ownership", "downstream_consumer"],
        as_index=False,
        observed=True,
    ).agg(
        Artifacts=("artifact_id", "size"),
        **{"Total size MB": ("size_mb", "sum")},
    ).round(3)
)

,ownership,downstream_consumer,Artifacts,Total size MB
0,4.7.1_owned,Showcase,6,209.449
1,upstream_reference,Audit / advanced Showcase,1,33.483
2,upstream_reference,Showcase,2,0.038
3,upstream_reference,Showcase / audit,9,57.552


**Findings.** The Showcase forecasting handoff now contains **six 4.7.1-owned app-facing artifacts**.

The detailed final-holdout record surface remains the evaluation and diagnostic layer. The longer forecast-history surface extends the frozen champion trajectories back to April 2025, while the compact Job, Taxi Zone, Temporal Bucket, and Date summaries continue to describe final-holdout performance.

The handoff also preserves the existing Chapter 4 reliability, failure-analysis, interpretation, and provenance evidence by reference rather than duplicating those upstream artifacts.

**4.7.1.5B — Run the final cross-handoff QA.**

These checks focus on methodological integrity rather than rediscovering model performance.

For the Pre-CP Reference, the QA verifies both the declared time boundaries and the physical assets that implement them: the 2024 family-selection matrix, the January 1–4 final-refit supplement, and the January 5 onward recursive application scaffold.

For the Showcase, the QA verifies that the final-holdout surface remains unchanged and that the longer forecast history preserves the complete 15-job system, the intended ordinary-validation/final-holdout boundary, and the frozen history row counts already validated in 4.7.1.3C.

The final QA also verifies that every 4.6.2 reliability caution remains `may_exclude_forecast = False`. That prevents a future consumer from turning holdout-derived error patterns into a retrospective eligibility rule.

Large row-level artifacts are checked through their validated metadata and compact audit markers where possible rather than rescanning millions of rows unnecessarily.

In [22]:
# ---------------------------------------------------------------------
# 4.7.1.5B — Final cross-handoff QA
# ---------------------------------------------------------------------

qa_rows = []

# ---------------------------------------------------------------------
# 4.2.4 selected-benchmark handoff boundary
# ---------------------------------------------------------------------
# 4.2.4 owns the row-level semantic validation. Here we verify its compact
# producer QA, manifest registration, and frozen schema without rescanning
# the complete prediction surface.

selected_pre_cp_required_columns = [
    "modeling_track",
    "stage",
    "taxi_zone_id",
    "metric",
    "forecast_sequence_id",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
    "horizon",
    "target_date",
    "target_observed_value",
    "target_label_available",
    "native_track_eligible",
    "all_baselines_available",
    "baseline_id",
    "baseline_label",
    "selection_metric",
    "selection_stage",
    "benchmark_prediction",
    "benchmark_prediction_available",
    "baseline_tournament_support_eligible",
    "selected_benchmark_support_eligible",
]

require_file(
    SELECTED_PRE_CP_BENCHMARK_PREDICTIONS_PATH,
    "Selected Pre-CP benchmark predictions",
)

selected_pre_cp_metadata = parquet_metadata(
    SELECTED_PRE_CP_BENCHMARK_PREDICTIONS_PATH
)

selected_pre_cp_actual_columns = set(
    pq.ParquetFile(
        SELECTED_PRE_CP_BENCHMARK_PREDICTIONS_PATH
    ).schema_arrow.names
)

selected_pre_cp_schema_match = (
    selected_pre_cp_actual_columns
    == set(selected_pre_cp_required_columns)
)

baseline_424_qa_passes, baseline_424_qa_df = qa_file_passes(
    BASELINE_FORECASTABILITY_QA_PATH
)

baseline_424_manifest_df = pd.read_parquet(
    BASELINE_FORECASTABILITY_MANIFEST_PATH
)

require_columns(
    baseline_424_manifest_df,
    ["asset", "path", "rows", "columns"],
    "4.2.4 baseline handoff manifest",
)

selected_424_manifest_rows = baseline_424_manifest_df.loc[
    baseline_424_manifest_df["asset"].eq(
        "selected_benchmark_predictions__pre_cp_reference"
    )
].copy()

selected_424_manifest_match = False

if len(selected_424_manifest_rows) == 1:
    selected_424_manifest_row = selected_424_manifest_rows.iloc[0]

    selected_424_manifest_match = (
        Path(str(selected_424_manifest_row["path"])).resolve()
        == SELECTED_PRE_CP_BENCHMARK_PREDICTIONS_PATH.resolve()
        and int(selected_424_manifest_row["rows"])
        == selected_pre_cp_metadata["rows"]
        and int(selected_424_manifest_row["columns"])
        == selected_pre_cp_metadata["columns"]
    )

# ---------------------------------------------------------------------
# Candidate-roster integrity
# ---------------------------------------------------------------------

family_counts = (
    counterfactual_candidate_registry_df["family"]
    .value_counts()
    .to_dict()
)

qa_rows.append(
    qa_row(
        "Four frozen forecasting families are packaged",
        set(family_counts) == set(FAMILY_ORDER),
        family_counts,
    )
)

qa_rows.append(
    qa_row(
        "Every family contributes all 15 Target × Horizon jobs",
        all(
            family_counts.get(family, 0) == EXPECTED_JOB_COUNT
            for family in FAMILY_ORDER
        ),
        family_counts,
    )
)

qa_rows.append(
    qa_row(
        "Chapter 5 Pre-CP winner remains intentionally unselected",
        not counterfactual_candidate_registry_df[
            "chapter5_pre_cp_selected"
        ].any(),
        (
            "selected candidates = "
            f"{int(counterfactual_candidate_registry_df['chapter5_pre_cp_selected'].sum())}"
        ),
    )
)

qa_rows.append(
    qa_row(
        "Post-policy Chapter 4 ranking is prohibited from Chapter 5 selection",
        not counterfactual_candidate_registry_df[
            "chapter4_ranking_allowed_for_chapter5_selection"
        ].any(),
        "All 60 candidates preserve the causal-selection boundary.",
    )
)

# ---------------------------------------------------------------------
# Executable model-contract integrity
# ---------------------------------------------------------------------
# Neural and Transformer both require the canonical dense mobility-history
# arrays to reconstruct literal sequence windows. This is observed historical
# input state, not fitted model state, so it may be reused only subject to the
# legal forecast-origin boundary enforced downstream.

shared_history_contract_rows = model_contract_manifest_df.loc[
    model_contract_manifest_df["family"].eq("neural")
    & model_contract_manifest_df["contract_role"].eq(
        "Shared Neural/Transformer mobility-history state"
    )
].copy()

shared_history_contract_valid = False

if len(shared_history_contract_rows) == 1:
    shared_history_contract_row = shared_history_contract_rows.iloc[0]

    shared_history_contract_valid = (
        len(model_contract_manifest_df) == 20
        and shared_history_contract_row["artifact_type"] == "npz"
        and shared_history_contract_row["chapter5_use"] == "reuse"
        and Path(shared_history_contract_row["path"]).resolve()
        == (
            INPUT_451_FINAL_DIR / "neural_mobility_history_state.npz"
        ).resolve()
    )

qa_rows.append(
    qa_row(
        "Executable model contracts include the shared sequence-history state",
        shared_history_contract_valid,
        (
            f"contract references = {len(model_contract_manifest_df)}; "
            f"shared history rows = {len(shared_history_contract_rows)}"
        ),
    )
)

# ---------------------------------------------------------------------
# Pre-CP Reference integrity
# ---------------------------------------------------------------------

qa_rows.append(qa_row(
    "Native Pre-CP population remains 1,039 sequences",
    len(pre_cp_sequence_support_registry_df) == EXPECTED_PRE_CP_NATIVE_SEQUENCES,
    f"rows = {len(pre_cp_sequence_support_registry_df):,}",
))

qa_rows.append(qa_row(
    "Primary cross-regime population remains 1,029 sequences",
    int(pre_cp_sequence_support_registry_df["primary_cross_regime"].sum())
    == EXPECTED_PRIMARY_CROSS_REGIME_SEQUENCES,
    (
        "primary rows = "
        f"{int(pre_cp_sequence_support_registry_df['primary_cross_regime'].sum()):,}"
    ),
))

qa_rows.append(qa_row(
    "Pre-CP benchmark registry contains all 15 jobs",
    (
        len(pre_cp_benchmark_registry_df) == EXPECTED_JOB_COUNT
        and not pre_cp_benchmark_registry_df.duplicated(
            ["modeling_track", "metric", "horizon"]
        ).any()
    ),
    f"rows = {len(pre_cp_benchmark_registry_df)}",
))

qa_rows.append(qa_row(
    "4.2.4 selected Pre-CP benchmark handoff is producer-validated",
    baseline_424_qa_passes and selected_424_manifest_match,
    (
        f"producer QA passed = {baseline_424_qa_passes}; "
        f"registered in 4.2.4 manifest = {selected_424_manifest_match}"
    ),
))

qa_rows.append(qa_row(
    "Selected Pre-CP benchmark surface exposes the frozen row-level contract",
    (
        selected_pre_cp_metadata["rows"] > 0
        and selected_pre_cp_metadata["columns"]
        == len(selected_pre_cp_required_columns)
        and selected_pre_cp_schema_match
    ),
    (
        f"rows = {selected_pre_cp_metadata['rows']:,}; "
        f"columns = {selected_pre_cp_metadata['columns']}; "
        f"schema match = {selected_pre_cp_schema_match}"
    ),
))

qa_rows.append(qa_row(
    "Pre-CP model selection ends on December 31, 2024",
    pd.Timestamp(
        contract["model_selection_target_end_date"]
    ).normalize() == PRE_CP_SELECTION_END_DATE,
    contract["model_selection_target_end_date"],
))

qa_rows.append(qa_row(
    "Final Pre-CP refit ends on January 4, 2025",
    (
        pd.Timestamp(contract["final_refit_end_date"]).normalize()
        == PRE_CP_FINAL_REFIT_END_DATE
        and int(contract["final_refit_end_observation_sequence_id"])
        == PRE_CP_FINAL_REFIT_SEQUENCE_ID
    ),
    (
        f"date = {contract['final_refit_end_date']}; "
        f"sequence = {int(contract['final_refit_end_observation_sequence_id'])}"
    ),
))

qa_rows.append(qa_row(
    "Counterfactual application begins January 5, 2025",
    pd.Timestamp(
        contract["counterfactual_application_start_date"]
    ).normalize() == COUNTERFACTUAL_START_DATE,
    contract["counterfactual_application_start_date"],
))


# ---------------------------------------------------------------------
# 4.2.3 final-refit supplement integrity
# ---------------------------------------------------------------------

forecasting_423_qa_passes, forecasting_423_qa_df = qa_file_passes(
    FINAL_FORECASTING_MATRIX_QA_PATH
)

forecasting_423_manifest_df = pd.read_parquet(
    FORECASTING_MATRIX_MANIFEST_PATH
)

require_columns(
    forecasting_423_manifest_df,
    ["input", "path", "rows", "columns"],
    "4.2.3 forecasting matrix manifest",
)

supplement_manifest_rows = forecasting_423_manifest_df.loc[
    forecasting_423_manifest_df["input"].eq(
        "Pre-CP final-refit training supplement"
    )
].copy()

chapter5_supplement_rows = pre_cp_handoff_manifest_df.loc[
    pre_cp_handoff_manifest_df["artifact_id"].eq(
        "pre_cp_final_refit_training_supplement"
    )
].copy()

supplement_metadata = parquet_metadata(
    PRE_CP_FINAL_REFIT_SUPPLEMENT_PATH
)

supplement_schema = pq.ParquetFile(
    PRE_CP_FINAL_REFIT_SUPPLEMENT_PATH
).schema_arrow

historical_pre_cp_schema = pq.ParquetFile(
    PRE_CP_FORECASTING_MATRIX_PATH
).schema_arrow

supplement_manifest_match = False

if len(supplement_manifest_rows) == 1:
    supplement_manifest_row = supplement_manifest_rows.iloc[0]

    supplement_manifest_match = (
        Path(str(supplement_manifest_row["path"])).resolve()
        == PRE_CP_FINAL_REFIT_SUPPLEMENT_PATH.resolve()
        and int(supplement_manifest_row["rows"])
        == supplement_metadata["rows"]
        and int(supplement_manifest_row["columns"])
        == supplement_metadata["columns"]
    )

qa_rows.append(qa_row(
    "4.2.3 final-refit supplement is producer-validated and registered",
    (
        forecasting_423_qa_passes
        and supplement_manifest_match
        and len(chapter5_supplement_rows) == 1
    ),
    (
        f"producer QA passed = {forecasting_423_qa_passes}; "
        f"4.2.3 manifest rows = {len(supplement_manifest_rows)}; "
        f"Chapter 5 manifest rows = {len(chapter5_supplement_rows)}"
    ),
))

qa_rows.append(qa_row(
    "Final-refit supplement matches the historical Pre-CP matrix schema",
    supplement_schema.equals(
        historical_pre_cp_schema,
        check_metadata=False,
    ),
    (
        f"historical columns = {len(historical_pre_cp_schema.names)}; "
        f"supplement columns = {len(supplement_schema.names)}"
    ),
))

supplement_audit_columns = [
    "modeling_track",
    "stage",
    "forecast_sequence_id",
    "metric",
    "origin_observation_sequence_id",
    "target_observation_sequence_id",
    "horizon",
    "target_date",
    "target_observed_value",
    "target_label_available",
]

supplement_audit_df = pd.read_parquet(
    PRE_CP_FINAL_REFIT_SUPPLEMENT_PATH,
    columns=supplement_audit_columns,
)

supplement_audit_df["target_date"] = pd.to_datetime(
    supplement_audit_df["target_date"]
).dt.normalize()

expected_supplement_rows = (
    EXPECTED_PRE_CP_NATIVE_SEQUENCES
    * 20
    * len(FORECAST_HORIZONS)
)

supplement_complete = (
    len(supplement_audit_df) == expected_supplement_rows
    and supplement_audit_df["forecast_sequence_id"].nunique()
    == EXPECTED_PRE_CP_NATIVE_SEQUENCES
    and supplement_audit_df["modeling_track"].eq("pre_cp_reference").all()
    and supplement_audit_df["stage"].eq("final_refit_only").all()
    and set(supplement_audit_df["horizon"].astype(int))
    == set(FORECAST_HORIZONS)
    and supplement_audit_df["target_date"].between(
        PRE_CP_REFIT_ONLY_START_DATE,
        PRE_CP_FINAL_REFIT_END_DATE,
        inclusive="both",
    ).all()
    and supplement_audit_df["target_label_available"].astype(bool).eq(
        supplement_audit_df["target_observed_value"].notna()
    ).all()
)

qa_rows.append(qa_row(
    "Final-refit supplement contains the complete January 1–4 supervised surface",
    supplement_complete,
    (
        f"rows = {len(supplement_audit_df):,}; "
        f"sequences = {supplement_audit_df['forecast_sequence_id'].nunique():,}; "
        f"horizons = {sorted(supplement_audit_df['horizon'].unique().tolist())}"
    ),
))

supplement_horizon_arithmetic_ok = (
    supplement_audit_df["target_observation_sequence_id"]
    .sub(supplement_audit_df["origin_observation_sequence_id"])
    .eq(supplement_audit_df["horizon"])
    .all()
)

supplement_reaches_freeze = (
    int(supplement_audit_df["target_observation_sequence_id"].max())
    == PRE_CP_FINAL_REFIT_SEQUENCE_ID
)

qa_rows.append(qa_row(
    "Final-refit supplement preserves horizon arithmetic and reaches January 4",
    supplement_horizon_arithmetic_ok and supplement_reaches_freeze,
    (
        "max target sequence = "
        f"{int(supplement_audit_df['target_observation_sequence_id'].max())}"
    ),
))

# The 2024 selection matrix and January 1–4 final-refit supplement occupy
# disjoint supervised target ranges.
first_refit_target_id = int(
    supplement_audit_df["target_observation_sequence_id"].min()
)

historical_refit_overlap_df = pd.read_parquet(
    PRE_CP_FORECASTING_MATRIX_PATH,
    columns=["target_observation_sequence_id"],
    filters=[
        (
            "target_observation_sequence_id",
            ">=",
            first_refit_target_id,
        ),
    ],
)

qa_rows.append(qa_row(
    "Final-refit rows are separate from the 2024 model-selection matrix",
    historical_refit_overlap_df.empty,
    (
        f"historical rows at/after target {first_refit_target_id} = "
        f"{len(historical_refit_overlap_df):,}"
    ),
))


template_metadata = parquet_metadata(
    PRE_CP_COUNTERFACTUAL_TEMPLATE_PATH
)

template_columns = set(
    pq.ParquetFile(
        PRE_CP_COUNTERFACTUAL_TEMPLATE_PATH
    ).schema_arrow.names
)

qa_rows.append(qa_row(
    "Pre-CP application template remains the frozen 7,023,640-row scaffold",
    template_metadata["rows"] == 7_023_640,
    f"rows = {template_metadata['rows']:,}",
))

qa_rows.append(qa_row(
    "Counterfactual template contains no observed future target",
    "target_observed_value" not in template_columns,
    (
        "target_observed_value present = "
        f"{'target_observed_value' in template_columns}"
    ),
))

# ---------------------------------------------------------------------
# Showcase integrity
# ---------------------------------------------------------------------

showcase_metadata = parquet_metadata(
    SHOWCASE_FORECAST_RECORD_SURFACE_PATH
)

qa_rows.append(
    qa_row(
        "Showcase record surface preserves benchmark-supported holdout",
        showcase_metadata["rows"] == EXPECTED_SHOWCASE_ROWS,
        f"rows = {showcase_metadata['rows']:,}",
    )
)

# ---------------------------------------------------------------------
# Longer Showcase forecast-history integrity
# ---------------------------------------------------------------------
# 3C already performed the expensive row-level validation. Final QA reads
# its compact audit so the notebook can verify the frozen history contract
# without rescanning nearly six million rows.

history_audit_current = large_artifact_matches_audit(
    SHOWCASE_FORECAST_HISTORY_SURFACE_PATH,
    SHOWCASE_FORECAST_HISTORY_AUDIT_PATH,
    SHOWCASE_HISTORY_CONTRACT_ID,
)

history_audit_df = pd.read_parquet(
    SHOWCASE_FORECAST_HISTORY_AUDIT_PATH
)

require_columns(
    history_audit_df,
    HISTORY_AUDIT_REQUIRED_COLUMNS,
    "Showcase forecast-history audit",
)

if len(history_audit_df) != 1:
    raise RuntimeError(
        "Showcase forecast-history audit must contain exactly one row."
    )

history_audit = history_audit_df.iloc[0]


# The first check freezes the physical artifact and the successful 3C row count.
history_contract_ok = (
    history_audit_current
    and int(history_audit["rows"]) == 5_980_119
    and int(history_audit["ordinary_validation_rows"]) == 4_558_683
    and int(history_audit["final_holdout_rows"]) == EXPECTED_SHOWCASE_ROWS
)

qa_rows.append(
    qa_row(
        "Showcase forecast history matches its frozen 3C contract",
        history_contract_ok,
        (
            f"total rows = {int(history_audit['rows']):,}; "
            "ordinary-validation rows = "
            f"{int(history_audit['ordinary_validation_rows']):,}; "
            "final-holdout rows = "
            f"{int(history_audit['final_holdout_rows']):,}"
        ),
    )
)


# Both evidence periods must retain the complete 15-job champion system.
history_jobs_ok = (
    int(history_audit["ordinary_validation_jobs"])
    == EXPECTED_JOB_COUNT
    and int(history_audit["final_holdout_jobs"])
    == EXPECTED_JOB_COUNT
)

qa_rows.append(
    qa_row(
        "Showcase forecast history preserves all 15 jobs in both periods",
        history_jobs_ok,
        (
            "ordinary-validation jobs = "
            f"{int(history_audit['ordinary_validation_jobs'])}; "
            "final-holdout jobs = "
            f"{int(history_audit['final_holdout_jobs'])}"
        ),
    )
)


# These dates describe the PERIOD envelopes only. They do not require every
# individual Target × Horizon to contain a row on each boundary date.
history_periods_ok = (
    pd.Timestamp(
        history_audit[
            "ordinary_validation_first_target_date"
        ]
    ).normalize()
    == SHOWCASE_HISTORY_VALIDATION_START_DATE
    and pd.Timestamp(
        history_audit[
            "ordinary_validation_last_target_date"
        ]
    ).normalize()
    == SHOWCASE_HISTORY_VALIDATION_END_DATE
    and pd.Timestamp(
        history_audit[
            "final_holdout_first_target_date"
        ]
    ).normalize()
    == SHOWCASE_HISTORY_HOLDOUT_START_DATE
    and pd.Timestamp(
        history_audit[
            "final_holdout_last_target_date"
        ]
    ).normalize()
    == SHOWCASE_HISTORY_HOLDOUT_END_DATE
)

qa_rows.append(
    qa_row(
        "Showcase forecast history preserves the two evaluation periods",
        history_periods_ok,
        (
            "ordinary validation = "
            f"{pd.Timestamp(history_audit['ordinary_validation_first_target_date']).date()} "
            "to "
            f"{pd.Timestamp(history_audit['ordinary_validation_last_target_date']).date()}; "
            "final holdout = "
            f"{pd.Timestamp(history_audit['final_holdout_first_target_date']).date()} "
            "to "
            f"{pd.Timestamp(history_audit['final_holdout_last_target_date']).date()}"
        ),
    )
)

qa_rows.append(
    qa_row(
        "Showcase job summary covers all 15 jobs",
        len(showcase_job_summary_df) == EXPECTED_JOB_COUNT,
        f"rows = {len(showcase_job_summary_df)}",
    )
)

qa_rows.append(
    qa_row(
        "Scale-aware Wins & Misses ranking is available for all jobs",
        (
            len(ranking_support_df) == EXPECTED_JOB_COUNT
            and ranking_support_df["rankable_win_rows"].gt(0).all()
            and ranking_support_df["rankable_miss_rows"].gt(0).all()
        ),
        f"jobs = {len(ranking_support_df)}",
    )
)


# ---------------------------------------------------------------------
# 4.6.2 diagnostic guidance must remain non-exclusionary
# ---------------------------------------------------------------------

reliability_guidance_df = pd.read_parquet(
    handoff_462_paths["reliability_guidance"]
)

require_columns(
    reliability_guidance_df,
    ["guidance_id", "may_exclude_forecast"],
    "4.6.2 reliability guidance",
)

qa_rows.append(
    qa_row(
        "4.6.2 reliability diagnostics do not become exclusion rules",
        not reliability_guidance_df[
            "may_exclude_forecast"
        ].astype(bool).any(),
        (
            "exclusion-permitted guidance rows = "
            f"{int(reliability_guidance_df['may_exclude_forecast'].astype(bool).sum())}"
        ),
    )
)


# ---------------------------------------------------------------------
# Manifest integrity
# ---------------------------------------------------------------------

manifest_paths = (
    list(pre_cp_handoff_manifest_df["path"])
    + list(showcase_handoff_manifest_df["path"])
)

missing_manifest_paths = [
    path for path in manifest_paths
    if not Path(path).exists()
]

qa_rows.append(
    qa_row(
        "Every advertised downstream artifact exists",
        len(missing_manifest_paths) == 0,
        (
            "missing paths = 0"
            if not missing_manifest_paths
            else missing_manifest_paths
        ),
    )
)


forecasting_system_handoff_qa_df = pd.DataFrame(qa_rows)

if not forecasting_system_handoff_qa_df["passed"].all():
    failed_checks = forecasting_system_handoff_qa_df.loc[
        ~forecasting_system_handoff_qa_df["passed"],
        ["check", "detail"],
    ]
    raise RuntimeError(
        "Final 4.7.1 QA failed:\n"
        + failed_checks.to_string(index=False)
    )

if WRITE_FINAL_OUTPUTS:
    atomic_write_parquet(
        forecasting_system_handoff_qa_df,
        FINAL_HANDOFF_QA_PATH,
    )


display(
    forecasting_system_handoff_qa_df.assign(
        Status=lambda frame: np.where(
            frame["passed"],
            "PASS",
            "FAIL",
        )
    )[
        ["check", "Status", "detail"]
    ].rename(
        columns={
            "check": "Check",
            "detail": "Detail",
        }
    )
)

print(
    "PASS — complete 4.7.1 forecasting handoff validated.\n"
    f"  QA checks: {len(forecasting_system_handoff_qa_df)}"
)

,Check,Status,Detail
0,Four frozen forecasting families are packaged,PASS,"{'classical': 15, 'neural': 15, 'transformer':..."
1,Every family contributes all 15 Target × Horiz...,PASS,"{'classical': 15, 'neural': 15, 'transformer':..."
2,Chapter 5 Pre-CP winner remains intentionally ...,PASS,selected candidates = 0
3,Post-policy Chapter 4 ranking is prohibited fr...,PASS,All 60 candidates preserve the causal-selectio...
4,Executable model contracts include the shared ...,PASS,contract references = 20; shared history rows = 1
5,"Native Pre-CP population remains 1,039 sequences",PASS,"rows = 1,039"
6,"Primary cross-regime population remains 1,029 ...",PASS,"primary rows = 1,029"
7,Pre-CP benchmark registry contains all 15 jobs,PASS,rows = 15
8,4.2.4 selected Pre-CP benchmark handoff is pro...,PASS,producer QA passed = True; registered in 4.2.4...
9,Selected Pre-CP benchmark surface exposes the ...,PASS,"rows = 5,680,584; columns = 21; schema match =..."


PASS — complete 4.7.1 forecasting handoff validated.
  QA checks: 28


**Findings.** All **28 final cross-handoff QA checks pass**.

The final package preserves the complete Chapter 4 forecasting and Chapter 5 handoff contracts while adding the longer Showcase forecast history as a separate, validated product.

The longitudinal history contains **5,980,119 rows** across all **15 Target × Horizon jobs**. Its **4,558,683 ordinary-validation rows** span April 5, 2025 through January 4, 2026, while its **1,421,436 final-holdout rows** span January 5 through March 31, 2026.

The existing final-holdout evaluation surface remains unchanged at **1,421,436 rows**, and the diagnostic reliability guidance remains non-exclusionary.

**4.7.1.5C — Freeze the single final Chapter 4 forecasting manifest.**

The final manifest gives downstream work one entry point for both uses of the forecasting system.

It points to the compact 4.7.1 products, while the nested Pre-CP and Showcase manifests preserve the larger producer-owned dependencies behind them. This avoids turning the final Chapter 4 handoff into another duplicate data warehouse.

In [23]:
# ---------------------------------------------------------------------
# 4.7.1.5C — Freeze the single final Chapter 4 forecasting manifest
# ---------------------------------------------------------------------

final_handoff_specs = [
    (
        "counterfactual_candidate_registry",
        COUNTERFACTUAL_CANDIDATE_REGISTRY_PATH,
        "Frozen four-family Chapter 5 candidate roster",
        "Chapter 5",
        "4.7.1_owned",
        "60 Family × Target × Horizon candidates; none pre-selected.",
    ),
    (
        "counterfactual_model_contract_manifest",
        MODEL_CONTRACT_MANIFEST_PATH,
        "Family-specific executable contract directory",
        "Chapter 5",
        "4.7.1_owned",
        "Authoritative model/input/preprocessing/output contract references.",
    ),
    (
        "chapter4_model_selection_provenance",
        CHAPTER4_SELECTION_PROVENANCE_PATH,
        "Ordinary Chapter 4 champion / runner-up provenance",
        "Chapter 5 / Showcase",
        "4.7.1_owned",
        "Preserves ordinary rankings without using them for causal selection.",
    ),
    (
        "pre_cp_reference_sequence_support_registry",
        PRE_CP_SEQUENCE_SUPPORT_REGISTRY_PATH,
        "Pre-CP support and causal-cohort contract",
        "Chapter 5",
        "4.7.1_owned",
        "Native support plus primary cross-regime membership.",
    ),
    (
        "pre_cp_reference_benchmark_registry",
        PRE_CP_BENCHMARK_REGISTRY_PATH,
        "Frozen Pre-CP benchmark roster",
        "Chapter 5",
        "4.7.1_owned",
        "One benchmark for each of the 15 Pre-CP jobs.",
    ),
    (
        "pre_cp_reference_handoff_manifest",
        PRE_CP_HANDOFF_MANIFEST_PATH,
        "Complete Pre-CP Reference directory",
        "Chapter 5",
        "4.7.1_owned",
        "Single entry point for model selection, refit, and recursive application.",
    ),
    (
        "showcase_forecast_record_surface",
        SHOWCASE_FORECAST_RECORD_SURFACE_PATH,
        "Benchmark-comparable forecast record surface",
        "Showcase",
        "4.7.1_owned",
        "Frozen 1,421,436-row system forecast and diagnostic surface.",
    ),
    (
        "showcase_forecast_history_surface",
        SHOWCASE_FORECAST_HISTORY_SURFACE_PATH,
        "Longitudinal champion forecast history",
        "Showcase",
        "4.7.1_owned",
        (
            "April 2025 through March 2026 champion-system history "
            "with ordinary-validation versus final-holdout provenance."
        ),
    ),
    (
        "showcase_forecasting_handoff_manifest",
        SHOWCASE_HANDOFF_MANIFEST_PATH,
        "Complete Showcase forecast directory",
        "Showcase",
        "4.7.1_owned",
        "Single entry point for row-level, summary, reliability, and failure evidence.",
    ),
    (
        "forecasting_system_handoff_qa",
        FINAL_HANDOFF_QA_PATH,
        "Final Chapter 4 forecasting QA",
        "Chapter 5 / Showcase",
        "4.7.1_owned",
        "Machine-readable integrity checks for the complete handoff.",
    ),
]

forecasting_system_handoff_manifest_df = pd.DataFrame([
    artifact_inventory_row(
        artifact_id,
        path,
        role,
        consumer,
        ownership,
        purpose,
    )
    for artifact_id, path, role, consumer, ownership, purpose
    in final_handoff_specs
])

if forecasting_system_handoff_manifest_df["artifact_id"].duplicated().any():
    raise RuntimeError(
        "Final Chapter 4 forecasting manifest contains duplicate IDs."
    )

if WRITE_FINAL_OUTPUTS:
    atomic_write_parquet(
        forecasting_system_handoff_manifest_df,
        FINAL_HANDOFF_MANIFEST_PATH,
    )


display(
    forecasting_system_handoff_manifest_df[
        [
            "artifact_id",
            "downstream_consumer",
            "rows",
            "size_mb",
            "purpose",
        ]
    ].rename(
        columns={
            "artifact_id": "Artifact",
            "downstream_consumer": "Consumer",
            "rows": "Rows",
            "size_mb": "Size MB",
            "purpose": "Purpose",
        }
    )
)

print(
    "PASS — Chapter 4 forecasting handoff frozen.\n"
    f"  Entry points: {len(forecasting_system_handoff_manifest_df)}\n"
    f"  Manifest: {FINAL_HANDOFF_MANIFEST_PATH}"
)

,Artifact,Consumer,Rows,Size MB,Purpose
0,counterfactual_candidate_registry,Chapter 5,60,0.108501,60 Family × Target × Horizon candidates; none ...
1,counterfactual_model_contract_manifest,Chapter 5,20,0.008308,Authoritative model/input/preprocessing/output...
2,chapter4_model_selection_provenance,Chapter 5 / Showcase,15,0.019495,Preserves ordinary rankings without using them...
3,pre_cp_reference_sequence_support_registry,Chapter 5,1039,0.020895,Native support plus primary cross-regime membe...
4,pre_cp_reference_benchmark_registry,Chapter 5,15,0.016054,One benchmark for each of the 15 Pre-CP jobs.
5,pre_cp_reference_handoff_manifest,Chapter 5,24,0.009653,"Single entry point for model selection, refit,..."
6,showcase_forecast_record_surface,Showcase,1421436,118.223045,"Frozen 1,421,436-row system forecast and diagn..."
7,showcase_forecast_history_surface,Showcase,5980119,90.882969,April 2025 through March 2026 champion-system ...
8,showcase_forecasting_handoff_manifest,Showcase,18,0.009073,"Single entry point for row-level, summary, rel..."
9,forecasting_system_handoff_qa,Chapter 5 / Showcase,28,0.004082,Machine-readable integrity checks for the comp...


PASS — Chapter 4 forecasting handoff frozen.
  Entry points: 10
  Manifest: /datasets/_deepnote_work/pipeline_data/4.7.1.final_tables/forecasting_system_handoff_manifest.parquet


**Findings.** The completed Chapter 4 forecasting handoff exposes **10 top-level entry points**.

Chapter 5 retains the frozen candidate, executable-model, Pre-CP support, benchmark, and recursive-application handoffs it needs.

The Showcase receives both the **1,421,436-row final-holdout evaluation surface** and the **5,980,119-row longitudinal champion history**, together with the compact forecasting and diagnostic handoffs.

Chapter 4 forecasting is therefore packaged for both longitudinal exploration and final-holdout evaluation without changing the frozen ordinary forecasting system.

### Section Summary

Notebook 4.7.1 has converted the completed Chapter 4 forecasting work into two explicit downstream products.

For **Chapter 5**, the handoff contains the frozen family candidates, executable contracts, Pre-CP validation and final-refit support, benchmark, and recursive no-CP application scaffold needed for the bounded counterfactual model-selection process.

For the **Showcase**, the handoff contains a **5,980,119-row longitudinal champion history** spanning April 5, 2025 through March 31, 2026 and a separate **1,421,436-row final-holdout evaluation surface** with its detailed reliability context. The compact Job, Taxi Zone, Temporal Bucket, and Date summaries remain final-holdout products.

The ordinary Chapter 4 champion map remains frozen. No final-holdout result, reliability finding, or post-policy forecast error is used to select the Chapter 5 counterfactual family.

### 4.7.1.8 — Freeze a Self-Contained Showcase Forecasting Package

The forecasting Showcase should not have to reach backward into earlier Chapter 4 folders just to recover compact diagnostic evidence.

The 4.7.1 package now contains the detailed final-holdout forecast surface, the longer champion-forecast history, and the final-holdout Job, Taxi Zone, Temporal Bucket, and Date summaries. It also resolves the authoritative 4.6.2 diagnostic handoff.

This portability step copies the small 4.6.2 evidence tables that may be useful to the Showcase into `4.7.1.final_tables` and records the complete transferable package in one manifest.

Two large upstream tables remain intentionally un-copied:

* `record_diagnostics` is already incorporated into the final-holdout record surface.
* The large 4.6.1 prediction surfaces are unnecessary because the reader-facing longitudinal history is now packaged directly by 4.7.1.

This changes file placement only. It does not rerun diagnostics, alter forecasts, or reopen model selection.

In [24]:
# ---------------------------------------------------------------------
# 4.7.1.8 — Materialize the portable Showcase forecasting package
# ---------------------------------------------------------------------

if not WRITE_FINAL_OUTPUTS:
    raise RuntimeError(
        "WRITE_FINAL_OUTPUTS must be True to freeze the portable "
        "Showcase forecasting package."
    )


# ---------------------------------------------------------------------
# 1) Compact upstream evidence worth carrying into the final package
# ---------------------------------------------------------------------
#
# record_diagnostics is intentionally absent. Its reader-facing fields
# were already merged into showcase_forecast_record_surface, so copying
# that 1.42M-row table again would add storage without adding evidence.

portable_462_ids = {
    "job_reliability_summary",
    "failure_mechanism_summary",
    "failure_case_studies",
    "failure_case_context",
    "model_interpretation_coverage",
    "tree_training_history_sensitivity",
    "reliability_guidance",
    "tree_feature_group_ablation",
}

optional_method_ids = {
    "model_interpretation_coverage",
    "tree_training_history_sensitivity",
    "tree_feature_group_ablation",
}

source_462_manifest_df = pd.read_parquet(
    FORECAST_EVALUATION_HANDOFF_462_PATH
)

required_manifest_columns = {
    "artifact_id",
    "path",
    "purpose",
}

missing_manifest_columns = sorted(
    required_manifest_columns
    - set(source_462_manifest_df.columns)
)

if missing_manifest_columns:
    raise KeyError(
        "4.6.2 handoff manifest is missing required columns: "
        f"{missing_manifest_columns}"
    )

source_462_manifest_df["artifact_id"] = (
    source_462_manifest_df["artifact_id"].astype(str)
)

available_462_ids = set(
    source_462_manifest_df["artifact_id"]
)

missing_462_ids = sorted(
    portable_462_ids - available_462_ids
)

if missing_462_ids:
    raise RuntimeError(
        "4.6.2 handoff is missing portable Showcase artifacts: "
        f"{missing_462_ids}"
    )


# ---------------------------------------------------------------------
# 2) Restart-safe physical copy helper
# ---------------------------------------------------------------------

def materialize_portable_copy(source_path, target_path):
    """
    Copy one upstream artifact into FINAL_DIR without rereading its rows.

    Existing same-size copies are reused so rerunning this closeout does
    not repeatedly copy large artifacts. A temporary file prevents an
    interrupted copy from being mistaken for a complete artifact.
    """
    source_path = Path(source_path)
    target_path = Path(target_path)

    if not source_path.exists():
        raise FileNotFoundError(
            f"Portable source does not exist: {source_path}"
        )

    source_size = source_path.stat().st_size

    if (
        target_path.exists()
        and target_path.stat().st_size == source_size
    ):
        return "reused"

    temp_path = target_path.with_name(
        f"{target_path.name}.copying"
    )

    if temp_path.exists():
        temp_path.unlink()

    shutil.copy2(source_path, temp_path)

    if temp_path.stat().st_size != source_size:
        temp_path.unlink(missing_ok=True)

        raise RuntimeError(
            "Portable copy failed byte-size validation: "
            f"{source_path.name}"
        )

    temp_path.replace(target_path)

    return "copied"


# ---------------------------------------------------------------------
# 3) Record native 4.7.1 Showcase artifacts
# ---------------------------------------------------------------------

native_showcase_specs = {
    "showcase_forecast_record_surface": (
        SHOWCASE_FORECAST_RECORD_SURFACE_PATH,
        "core",
        "Benchmark-comparable row-level forecast surface.",
    ),
    "showcase_forecast_history_surface": (
        SHOWCASE_FORECAST_HISTORY_SURFACE_PATH,
        "core",
        (
            "Longitudinal champion-system forecast history "
            "with evaluation-period provenance."
        ),
    ),
    "showcase_forecast_job_summary": (
        SHOWCASE_JOB_SUMMARY_PATH,
        "core",
        "Target × Horizon forecast-performance summary.",
    ),
    "showcase_forecast_zone_summary": (
        SHOWCASE_ZONE_SUMMARY_PATH,
        "core",
        "Target × Horizon × Taxi Zone reliability summary.",
    ),
    "showcase_forecast_temporal_bucket_summary": (
        SHOWCASE_TEMPORAL_BUCKET_SUMMARY_PATH,
        "core",
        "Temporal-bucket forecast-reliability summary.",
    ),
    "showcase_forecast_date_summary": (
        SHOWCASE_DATE_SUMMARY_PATH,
        "core",
        "Date-level forecast-reliability summary.",
    ),
    "showcase_forecasting_handoff_manifest": (
        SHOWCASE_HANDOFF_MANIFEST_PATH,
        "provenance",
        "Original 4.7.1 forecasting Showcase handoff.",
    ),
    "forecasting_system_handoff_qa": (
        FINAL_HANDOFF_QA_PATH,
        "provenance",
        "Final Chapter 4 forecasting handoff QA.",
    ),
}

portable_records = []

for artifact_id, (
    artifact_path,
    showcase_scope,
    purpose,
) in native_showcase_specs.items():
    artifact_path = Path(artifact_path)

    if not artifact_path.exists():
        raise FileNotFoundError(
            f"Required 4.7.1 Showcase artifact is missing: "
            f"{artifact_path}"
        )

    portable_records.append({
        "artifact_id": artifact_id,
        "source_notebook": "4.7.1",
        "showcase_scope": showcase_scope,
        "source_path": str(artifact_path),
        "portable_path": str(artifact_path),
        "copy_status": "native",
        "size_mb": (
            artifact_path.stat().st_size / (1024 ** 2)
        ),
        "purpose": purpose,
    })


# ---------------------------------------------------------------------
# 4) Promote compact 4.6.2 evidence into 4.7.1.final_tables
# ---------------------------------------------------------------------

promotion_df = (
    source_462_manifest_df.loc[
        source_462_manifest_df["artifact_id"]
        .isin(portable_462_ids)
    ]
    .copy()
    .sort_values("artifact_id")
)

for row in promotion_df.itertuples(index=False):
    source_path = Path(str(row.path))
    target_path = FINAL_DIR / source_path.name

    copy_status = materialize_portable_copy(
        source_path,
        target_path,
    )

    showcase_scope = (
        "optional_methods"
        if row.artifact_id in optional_method_ids
        else "core"
    )

    portable_records.append({
        "artifact_id": str(row.artifact_id),
        "source_notebook": "4.6.2",
        "showcase_scope": showcase_scope,
        "source_path": str(source_path),
        "portable_path": str(target_path),
        "copy_status": copy_status,
        "size_mb": (
            target_path.stat().st_size / (1024 ** 2)
        ),
        "purpose": str(row.purpose),
    })


# ---------------------------------------------------------------------
# 5) Freeze one portable transfer manifest
# ---------------------------------------------------------------------

showcase_portable_manifest_df = (
    pd.DataFrame(portable_records)
    .sort_values(
        ["showcase_scope", "source_notebook", "artifact_id"]
    )
    .reset_index(drop=True)
)

if showcase_portable_manifest_df["artifact_id"].duplicated().any():
    duplicate_ids = (
        showcase_portable_manifest_df.loc[
            showcase_portable_manifest_df[
                "artifact_id"
            ].duplicated(keep=False),
            "artifact_id",
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    raise RuntimeError(
        "Portable forecasting manifest contains duplicate IDs: "
        f"{sorted(duplicate_ids)}"
    )

missing_portable_paths = [
    path
    for path in showcase_portable_manifest_df["portable_path"]
    if not Path(path).exists()
]

if missing_portable_paths:
    raise RuntimeError(
        "Portable forecasting package contains missing files: "
        f"{missing_portable_paths}"
    )

manifest_temp_path = (
    SHOWCASE_PORTABLE_MANIFEST_PATH.with_name(
        f"{SHOWCASE_PORTABLE_MANIFEST_PATH.name}.tmp"
    )
)

showcase_portable_manifest_df.to_parquet(
    manifest_temp_path,
    index=False,
)
manifest_temp_path.replace(
    SHOWCASE_PORTABLE_MANIFEST_PATH
)

display(
    showcase_portable_manifest_df[
        [
            "artifact_id",
            "source_notebook",
            "showcase_scope",
            "copy_status",
            "size_mb",
            "portable_path",
        ]
    ]
)

print(
    "\nPortable forecasting package frozen:"
    f"\n  artifacts listed = "
    f"{len(showcase_portable_manifest_df):,}"
    f"\n  core artifacts = "
    f"{showcase_portable_manifest_df['showcase_scope'].eq('core').sum():,}"
    f"\n  optional-method artifacts = "
    f"{showcase_portable_manifest_df['showcase_scope'].eq('optional_methods').sum():,}"
    f"\n  total size = "
    f"{showcase_portable_manifest_df['size_mb'].sum() / 1024:.3f} GB"
    f"\n  manifest = {SHOWCASE_PORTABLE_MANIFEST_PATH}",
    flush=True,
)

,artifact_id,source_notebook,showcase_scope,copy_status,size_mb,portable_path
0,failure_case_context,4.6.2,core,reused,0.071398,/datasets/_deepnote_work/pipeline_data/4.7.1.f...
1,failure_case_studies,4.6.2,core,reused,0.058536,/datasets/_deepnote_work/pipeline_data/4.7.1.f...
2,failure_mechanism_summary,4.6.2,core,reused,0.009109,/datasets/_deepnote_work/pipeline_data/4.7.1.f...
3,job_reliability_summary,4.6.2,core,reused,0.022015,/datasets/_deepnote_work/pipeline_data/4.7.1.f...
4,reliability_guidance,4.6.2,core,reused,0.003963,/datasets/_deepnote_work/pipeline_data/4.7.1.f...
5,showcase_forecast_date_summary,4.7.1,core,native,0.099071,/datasets/_deepnote_work/pipeline_data/4.7.1.f...
6,showcase_forecast_history_surface,4.7.1,core,native,90.882969,/datasets/_deepnote_work/pipeline_data/4.7.1.f...
7,showcase_forecast_job_summary,4.7.1,core,native,0.003560,/datasets/_deepnote_work/pipeline_data/4.7.1.f...
8,showcase_forecast_record_surface,4.7.1,core,native,118.223045,/datasets/_deepnote_work/pipeline_data/4.7.1.f...
9,showcase_forecast_temporal_bucket_summary,4.7.1,core,native,0.015117,/datasets/_deepnote_work/pipeline_data/4.7.1.f...



Portable forecasting package frozen:
  artifacts listed = 16
  core artifacts = 11
  optional-method artifacts = 3
  total size = 0.205 GB
  manifest = /datasets/_deepnote_work/pipeline_data/4.7.1.final_tables/showcase_forecasting_portable_manifest.parquet


### Findings

The Chapter 4 Showcase forecasting package is physically self-contained.

The six reader-facing 4.7.1 forecast products are the detailed final-holdout record surface, the longer forecast-history surface, and the four compact final-holdout summaries. Compact diagnostic evidence from 4.6.2 travels with them in `4.7.1.final_tables`.

The large `record_diagnostics` table is not duplicated because its reader-facing fields are already incorporated into `showcase_forecast_record_surface.parquet`.

`showcase_forecasting_portable_manifest.parquet` is the transfer checklist for the forecasting side of the Showcase.

## Close

Notebook 4.7.1 closes Chapter 4 forecasting.

The forecasting pipeline has a clean boundary between **forecast evaluation** and **counterfactual construction**.

Chapter 4's 15 Target × Horizon champions remain frozen for ordinary forecasting interpretation and Showcase reporting.

Chapter 5 receives the frozen family configurations and Pre-CP contracts needed for its bounded counterfactual model-selection process. That process remains separate from Chapter 4's post-policy holdout results.

The Showcase receives two complementary forecasting products: a **longitudinal champion history** for exploring forecast behavior from April 2025 through March 2026, and a **final-holdout evaluation surface** for judging out-of-sample forecast performance and reliability.

Chapter 4 therefore ends with its forecasting outputs packaged for downstream use without changing a frozen model decision.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>